In [ ]:
import os

import matplotlib.pyplot as plt
import numpy as np
import scipy.optimize

import mrfitty
from mrfitty.base import (
    AdaptiveEnergyRangeBuilder,
    InterpolatedReferenceSpectraSet,
    ReferenceSpectrum,
)

%matplotlib inline

In [ ]:
import fnmatch


def filter_spectra_by_name(spectra_list, *patterns):
    """Return spectra whose file_name matches any of the glob-style patterns."""
    matches = [s for s in spectra_list if any(fnmatch.fnmatch(s.file_name, p) for p in patterns)]
    if not matches:
        raise ValueError(
            f"No spectra matched the given pattern(s): {patterns!r}. "
            f"Available file names: {[s.file_name for s in spectra_list]}"
        )
    return matches


In [ ]:
src_path, _ = os.path.split(mrfitty.__path__[0])
sample_data_dir_path = os.path.join(src_path, 'example', 'arsenic')
print('sample data is installed at "{}"'.format(sample_data_dir_path))
os.path.exists(sample_data_dir_path)

In [ ]:
reference_spectra_glob = os.path.join(sample_data_dir_path, 'reference/*.e')
print('reference spectra glob: {}'.format(reference_spectra_glob))
sample_spectra_glob = os.path.join(sample_data_dir_path, 'unknown/*.e')
print('sample spectra glob: {}'.format(sample_spectra_glob))

In [ ]:
reference_spectra_list = sorted(list(ReferenceSpectrum.read_all([reference_spectra_glob])[0]), key=lambda s: s.file_name)
print('reference spectra file count: {}'.format(len(reference_spectra_list)))
sample_spectra_list = sorted(list(ReferenceSpectrum.read_all([sample_spectra_glob])[0]), key=lambda s: s.file_name)
print('sample spectra file count: {}'.format(len(sample_spectra_list)))

In [ ]:
def fit_nnls(A, b):
    coef, _ = scipy.optimize.nnls(A, b)
    fitted = A @ coef
    residuals = fitted - b
    return coef, fitted, residuals


In [ ]:
def fit_ols(A, b):
    coef = np.linalg.solve(A.T @ A, A.T @ b)
    fitted = A @ coef
    residuals = fitted - b
    return coef, fitted, residuals


In [ ]:
import statsmodels.api as sm


def fit_ols_with_statistics(A, b):
    result = sm.OLS(b, A).fit()
    coef = result.params
    fitted = result.fittedvalues
    residuals = fitted - b
    return coef, fitted, residuals, result


In [ ]:
unknown_spectrum = filter_spectra_by_name(sample_spectra_list, "OTT3_55*")[0]
print(f'unknown spectrum: {unknown_spectrum.file_name}')
reference_spectra = filter_spectra_by_name(
    reference_spectra_list,
    "Arsenopyrite_Jul*", "orpiment_all*", "arsenate*_diop*") 
ref_names = tuple([r.file_name for r in reference_spectra])
print(f"reference spectra:\n  {'\n  '.join(ref_names)}")

In [ ]:
# Interpolation methods for resampling reference spectra onto the sample's energy grid.
#
# make_interp_spline is scipy's current spline-construction API -- the modern
# replacement for the legacy interp1d / splrep interface -- and covers both methods
# we want through its degree parameter k, so linear and cubic differ only in one
# argument rather than in which function is called.
#
# Each factory takes (energies, norm) and returns a callable evaluated at arbitrary
# energies, so the *factory* is the unit passed to
# interpolate_references_at_sample_energies.
#
# make_cubic_spline_interpolant is the default there, and it is not a change in
# behavior: ReferenceSpectrum.__init__ (mrfitty/base.py) builds an
# InterpolatedUnivariateSpline, which is also a k=3 interpolating spline, and the two
# agree to ~1e-15 in norm units on this data. Building the interpolant here rather
# than reading the one stored on the spectrum is what makes the method selectable.
from scipy.interpolate import make_interp_spline


def make_linear_interpolant(energies, norm):
    """Piecewise-linear interpolant through (energies, norm)."""
    return make_interp_spline(energies, norm, k=1)


def make_cubic_spline_interpolant(energies, norm):
    """Interpolating cubic spline through (energies, norm)."""
    return make_interp_spline(energies, norm, k=3)

In [ ]:
def interpolate_references_at_sample_energies(
    reference_spectra, sample_spectrum, make_interpolant=make_cubic_spline_interpolant,
):
    """Interpolate reference spectra onto the sample spectrum's energy grid.

    The usable energy range is the intersection of the sample spectrum's range
    and every reference spectrum's range, so no extrapolation occurs.  Each
    reference is resampled by building an interpolant through its own measured
    energies and evaluating it at the sample energies.

    Parameters
    ----------
    reference_spectra : list of ReferenceSpectrum
        Pool of reference spectra to interpolate.  Each must expose a
        ``data_df`` attribute indexed by energy (eV) with a ``norm`` column.
    sample_spectrum : Spectrum or ReferenceSpectrum
        The unknown spectrum to be fitted.  Must expose a ``data_df`` attribute
        whose index contains energy values (eV) and whose ``norm`` column
        contains the normalized fluorescence values used as the regression
        response vector.
    make_interpolant : callable, optional
        Factory called as ``make_interpolant(energies, norm)`` returning a
        callable that evaluates the reference at arbitrary energies.  Defaults
        to ``make_cubic_spline_interpolant``, matching the cubic spline
        ``ReferenceSpectrum`` builds internally.  Pass
        ``make_linear_interpolant`` to resample linearly instead.  This choice
        sits upstream of the design matrix, the fit coefficients, and the
        prediction error, so it can affect which reference combination is
        selected -- see the interpolation method comparison at the end of this
        notebook.

    Returns
    -------
    valid_energies : ndarray, shape (n,)
        Energy values (eV) at which interpolation was performed — the
        intersection of the sample spectrum's range and all reference ranges.
        Depends only on the measured energy ranges, not on ``make_interpolant``.
    A : ndarray, shape (n, n_refs)
        Design matrix A for linear regression: column i holds reference i
        interpolated at ``valid_energies``.
    b : ndarray, shape (n,)
        Sample spectrum normalized fluorescence values at ``valid_energies``.
        This is the response vector for linear regression against A.
    low_limiters : list
        The spectrum object(s) whose lower bound sets ``valid_energies[0]``
        (the highest lower bound).  More than one when several tie exactly.
        The sample spectrum is a candidate alongside the references.
    high_limiters : list
        The spectrum object(s) whose upper bound sets ``valid_energies[-1]``
        (the lowest upper bound).  More than one when several tie exactly.
    """
    energies = sample_spectrum.data_df.index.values
    print(f'sample_spectrum: {sample_spectrum.file_name}')
    print(f'energy range: {energies[0]:.2f}–{energies[-1]:.2f} eV ({len(energies)} points)')
    print(f'references: {len(reference_spectra)}')
    print(f'interpolation: {make_interpolant.__name__}')

    # restrict to energies covered by the sample_spectrum AND every reference to avoid extrapolation.
    # collect (spectrum, low, high) for the sample and every reference so we can report and return
    # which spectra limit each end of the common range.
    spectrum_ranges = [
        (s, s.data_df.index.values[0], s.data_df.index.values[-1])
        for s in (sample_spectrum, *reference_spectra)
    ]

    # energy_min is set by the highest lower bound; energy_max by the lowest upper bound.
    # report every spectrum tied at each limiting energy, not just the first.
    energy_min = max(low for _, low, _ in spectrum_ranges)
    energy_max = min(high for _, _, high in spectrum_ranges)
    low_limiters = [s for s, low, _ in spectrum_ranges if low == energy_min]
    high_limiters = [s for s, _, high in spectrum_ranges if high == energy_max]

    print(f'lowest valid energy {energy_min:.2f} eV limited by '
          f'{", ".join(s.file_name for s in low_limiters)}')
    print(f'highest valid energy {energy_max:.2f} eV limited by '
          f'{", ".join(s.file_name for s in high_limiters)}')

    valid_mask = (energies >= energy_min) & (energies <= energy_max)
    valid_energies = energies[valid_mask]
    n_excluded = (~valid_mask).sum()
    if n_excluded:
        print(f'excluded {n_excluded} energies outside common range '
              f'({energy_min:.2f}–{energy_max:.2f} eV)')
    print(f'interpolating at {len(valid_energies)} energies '
          f'({valid_energies[0]:.2f}–{valid_energies[-1]:.2f} eV)')

    A = np.zeros((len(valid_energies), len(reference_spectra)))
    for i, ref in enumerate(reference_spectra):
        # build the interpolant from the reference's own measured points rather than
        # using a pre-built one, so make_interpolant actually selects the method
        reference_interpolant = make_interpolant(
            ref.data_df.index.values, ref.data_df['norm'].values,
        )
        A[:, i] = reference_interpolant(valid_energies)
        print(f'  {ref.file_name}: norm [{A[:, i].min():.4f}, {A[:, i].max():.4f}]')

    b = sample_spectrum.data_df['norm'].values[valid_mask]

    return valid_energies, A, b, low_limiters, high_limiters

## Tests for `interpolate_references_at_sample_energies`

The tests below exercise the function with tiny, hand-built synthetic spectra so
that every expected value is known exactly.  They cover:

- the common energy range being the intersection of the sample and all references,
- the shapes and values of the returned design matrix `A` and response vector `b`,
- correct identification of the low- and high-energy limiting spectra,
- reporting **all** spectra tied at a limiting energy,
- the sample spectrum itself acting as a limiter,
- nothing being excluded when the ranges coincide.

They are written as plain `pytest` functions (no fixtures, no arguments), so they
are collected automatically by `pytest`; the final cell also runs them directly
inside the notebook.

In [ ]:
# ---------------------------------------------------------------------------
# Test fixtures: a minimal stand-in for ReferenceSpectrum / Spectrum
# ---------------------------------------------------------------------------
# interpolate_references_at_sample_energies() only ever touches two things on
# the spectra it is handed:
#   * .file_name              - a label used in the printed / returned report
#   * .data_df                - a pandas DataFrame indexed by energy (eV) with a
#                               'norm' column of normalized fluorescence values
# It builds its own interpolant from .data_df via the make_interpolant argument,
# so a real spectrum's pre-built .interpolant is never read and the fixture does
# not need to supply one.
# FakeSpectrum supplies exactly those two, so the tests can drive the function
# with tiny, fully-controlled inputs instead of reading real .e files from disk.
import numpy as np
import pandas as pd


class FakeSpectrum:
    def __init__(self, file_name, energies, norm):
        # store energy/norm as float arrays so integer test grids behave like real data
        energies = np.asarray(energies, dtype=float)
        norm = np.asarray(norm, dtype=float)
        self.file_name = file_name
        # energy is the index; 'norm' is the single column the function reads
        self.data_df = pd.DataFrame({'norm': norm}, index=energies)

In [ ]:
def test_returns_intersection_range_shapes_and_values():
    # Sample grid spans 0..10 on integer eV; its 'norm' is a simple ramp (10*E)
    # so the expected response vector b is trivial to read off.
    sample = FakeSpectrum('sample', np.arange(0, 11), np.arange(0, 11) * 10.0)

    # The references sit on grids shifted a fraction of an eV off the sample grid,
    # so none of their nodes line up with the sample energies and the function
    # must genuinely interpolate them.  Each reference's 'norm' is still a linear
    # function of energy, and linear interpolation reproduces a linear function
    # exactly, so the expected values at the sample energies stay simple.
    offset = 0.3
    ref_a_energies = np.arange(2, 11) - offset   # 1.7 .. 9.7 -> highest lower bound (1.7 eV)
    ref_a = FakeSpectrum('ref_a', ref_a_energies, 2.0 * ref_a_energies)   # norm = 2*E
    ref_b_energies = np.arange(0, 9) + offset    # 0.3 .. 8.3 -> lowest upper bound (8.3 eV)
    ref_b = FakeSpectrum('ref_b', ref_b_energies, ref_b_energies + 1.0)   # norm = E + 1

    valid_energies, A, b, low_limiters, high_limiters = \
        interpolate_references_at_sample_energies([ref_a, ref_b], sample)

    # Common range is the intersection [max(lows), min(highs)] = [1.7, 8.3]; the
    # valid energies are the SAMPLE grid points falling inside it, i.e. 2..8.
    np.testing.assert_array_equal(valid_energies, np.array([2, 3, 4, 5, 6, 7, 8], dtype=float))

    # A: one row per valid energy, one column per reference, in the input order.
    assert A.shape == (7, 2)
    # Column 0 is ref_a interpolated onto valid_energies (2*E), column 1 is ref_b (E+1).
    # These hold despite the offset grids because linear interp of linear data is exact.
    np.testing.assert_allclose(A[:, 0], 2.0 * valid_energies)
    np.testing.assert_allclose(A[:, 1], valid_energies + 1.0)

    # b is the sample's 'norm' (10*E) sampled at the valid energies - no interpolation.
    np.testing.assert_allclose(b, 10.0 * valid_energies)

In [ ]:
def test_identifies_low_and_high_limiters():
    # 'norm' values are irrelevant here, so use zeros; only the energy bounds matter.
    sample = FakeSpectrum('sample', np.arange(0, 11), np.zeros(11))
    ref_a = FakeSpectrum('ref_a', np.arange(2, 11), np.zeros(9))   # starts latest (2 eV)
    ref_b = FakeSpectrum('ref_b', np.arange(0, 9), np.zeros(9))    # ends earliest (8 eV)

    _, _, _, low_limiters, high_limiters = \
        interpolate_references_at_sample_energies([ref_a, ref_b], sample)

    # ref_a alone has the highest lower bound, so it alone limits the low end;
    # ref_b alone has the lowest upper bound, so it alone limits the high end.
    assert low_limiters == [ref_a]
    assert high_limiters == [ref_b]

In [ ]:
def test_reports_all_tied_limiters():
    sample = FakeSpectrum('sample', np.arange(0, 11), np.zeros(11))
    # ref_a and ref_c share the exact same starting energy (2 eV) ...
    ref_a = FakeSpectrum('ref_a', np.arange(2, 11), np.zeros(9))
    ref_c = FakeSpectrum('ref_c', np.arange(2, 11), np.zeros(9))
    # ... and ref_b and ref_d share the exact same ending energy (8 eV).
    ref_b = FakeSpectrum('ref_b', np.arange(0, 9), np.zeros(9))
    ref_d = FakeSpectrum('ref_d', np.arange(0, 9), np.zeros(9))

    _, _, _, low_limiters, high_limiters = \
        interpolate_references_at_sample_energies([ref_a, ref_c, ref_b, ref_d], sample)

    # Every spectrum tied at a limiting energy is reported (not just the first),
    # in the order the function scans them: sample first, then the references as given.
    assert low_limiters == [ref_a, ref_c]
    assert high_limiters == [ref_b, ref_d]

In [ ]:
def test_sample_spectrum_can_be_the_limiter():
    # The sample is NARROWER than both references (3..7 vs 0..10), so the sample
    # itself limits BOTH ends and none of its grid points are excluded.
    sample = FakeSpectrum('sample', np.arange(3, 8), np.zeros(5))
    ref_a = FakeSpectrum('ref_a', np.arange(0, 11), np.zeros(11))
    ref_b = FakeSpectrum('ref_b', np.arange(0, 11), np.zeros(11))

    valid_energies, A, b, low_limiters, high_limiters = \
        interpolate_references_at_sample_energies([ref_a, ref_b], sample)

    # No sample energies fall outside the common range, so the full grid is kept.
    np.testing.assert_array_equal(valid_energies, np.arange(3, 8, dtype=float))
    assert A.shape == (5, 2)
    # The sample is the tightest spectrum at each end, so it is the sole limiter.
    assert low_limiters == [sample]
    assert high_limiters == [sample]

In [ ]:
def test_identical_ranges_exclude_nothing():
    # When the sample and reference share the same grid, the whole grid is valid
    # and the design matrix has exactly one column.
    grid = np.arange(0, 6)
    sample = FakeSpectrum('sample', grid, np.arange(0, 6) * 1.0)
    ref_a = FakeSpectrum('ref_a', grid, np.arange(0, 6) * 1.0)

    valid_energies, A, b, low_limiters, high_limiters = \
        interpolate_references_at_sample_energies([ref_a], sample)

    np.testing.assert_array_equal(valid_energies, grid.astype(float))
    assert A.shape == (6, 1)
    # With identical bounds, both the sample and the reference tie at each edge.
    assert low_limiters == [sample, ref_a]
    assert high_limiters == [sample, ref_a]

In [ ]:
def test_interpolation_method_is_selectable():
    # Every other fixture in this file uses linear (or zero) 'norm' data, which both
    # a linear and a cubic interpolant reproduce exactly -- so those tests cannot tell
    # the two methods apart. This one uses data with genuine curvature.
    #
    # The reference sits on a COARSE grid (every 2 eV) and is sampled on a FINER grid
    # (every 1 eV), so half the sample energies fall strictly between reference nodes
    # and must actually be interpolated. That is the same situation as the real data,
    # where several references are measured every 1.05 eV against a 0.5 eV sample grid.
    def cubic_norm(energy):
        return 0.01 * energy ** 3 - 0.2 * energy ** 2 + energy + 1.0

    sample_energies = np.arange(0, 21, 1)
    sample = FakeSpectrum('sample', sample_energies, np.zeros(len(sample_energies)))
    reference_energies = np.arange(0, 21, 2)
    reference = FakeSpectrum('curved_ref', reference_energies, cubic_norm(reference_energies))

    _, A_cubic, _, _, _ = interpolate_references_at_sample_energies(
        [reference], sample, make_interpolant=make_cubic_spline_interpolant,
    )
    _, A_linear, _, _, _ = interpolate_references_at_sample_energies(
        [reference], sample, make_interpolant=make_linear_interpolant,
    )

    # An interpolating cubic spline through samples of a cubic polynomial reproduces
    # that polynomial exactly, so the cubic arm is right to floating-point tolerance.
    np.testing.assert_allclose(A_cubic[:, 0], cubic_norm(sample_energies), atol=1e-10)

    # The linear arm chords across each 2 eV gap and so must NOT match, and the
    # disagreement has to be large enough to matter rather than a rounding artifact.
    assert not np.allclose(A_linear[:, 0], cubic_norm(sample_energies), atol=1e-10)
    assert np.abs(A_linear[:, 0] - A_cubic[:, 0]).max() > 0.05

    # Both methods interpolate rather than extrapolate, so they agree exactly wherever
    # a sample energy coincides with a reference node -- the difference is confined to
    # the points between nodes.
    on_node = np.isin(sample_energies, reference_energies)
    np.testing.assert_allclose(A_linear[on_node, 0], A_cubic[on_node, 0], atol=1e-10)
    assert np.abs(A_linear[~on_node, 0] - A_cubic[~on_node, 0]).max() > 0.05

In [ ]:
def test_default_interpolation_is_cubic():
    # Pins the default. Everything in this notebook that was produced before
    # make_interpolant existed -- the v1-v5 holdout comparison, the fits, the
    # prediction error tables -- was computed with the cubic spline
    # ReferenceSpectrum builds internally, so the default has to stay cubic for
    # those results to remain reproducible.
    def cubic_norm(energy):
        return 0.01 * energy ** 3 - 0.2 * energy ** 2 + energy + 1.0

    sample_energies = np.arange(0, 21, 1)
    sample = FakeSpectrum('sample', sample_energies, np.zeros(len(sample_energies)))
    reference_energies = np.arange(0, 21, 2)
    reference = FakeSpectrum('curved_ref', reference_energies, cubic_norm(reference_energies))

    _, A_default, _, _, _ = interpolate_references_at_sample_energies([reference], sample)
    _, A_cubic, _, _, _ = interpolate_references_at_sample_energies(
        [reference], sample, make_interpolant=make_cubic_spline_interpolant,
    )

    np.testing.assert_array_equal(A_default, A_cubic)

In [ ]:
# Under pytest these test_* functions are collected and run automatically.
# Inside the notebook we invoke each one directly and report pass/fail so the
# suite can be exercised interactively as well.
_test_fns = [obj for name, obj in sorted(globals().items())
             if name.startswith('test_') and callable(obj)]
for _fn in _test_fns:
    _fn()
    print(f'PASSED: {_fn.__name__}')
print(f'\n{len(_test_fns)} tests passed')

### Visualizing the interpolated design matrix

`plot_interpolated_references` draws the output of
`interpolate_references_at_sample_energies` on a single axis: the sample response
vector `b` and every interpolated reference column of `A`, all against
`valid_energies`. The edges of the common energy range are marked with dashed
lines annotated with the spectra that limit them.

In [ ]:
def plot_interpolated_references(
    valid_energies, A, b, low_limiters, high_limiters,
    reference_spectra, sample_spectrum, ax=None,
):
    """Visualize the output of interpolate_references_at_sample_energies.

    Built to stay legible from 3 references to 30+.  The sample is bold black;
    references are colored by the *role* they play, not one hue each, so the
    legend never explodes:
      * references that limit the low edge  -> blue
      * references that limit the high edge -> orange
      * references that limit both edges    -> green
      * every other reference               -> faint gray context (one legend row)
    Color is never the only cue: the legend names each role and its count, and
    the red dashed bound lines name the limiting spectra (or their count when many
    tie at the same energy).

    Parameters
    ----------
    valid_energies, A, b, low_limiters, high_limiters
        The five values returned by interpolate_references_at_sample_energies.
    reference_spectra : list of ReferenceSpectrum
        The same reference pool passed to that function; ``A``'s columns follow
        this order.
    sample_spectrum : Spectrum or ReferenceSpectrum
        The sample whose response vector ``b`` is plotted.
    ax : matplotlib Axes, optional
        Axis to draw on. A new figure/axis is created when omitted.

    Returns
    -------
    ax : matplotlib Axes
        The axis the plot was drawn on.
    """
    from collections import Counter
    from matplotlib.lines import Line2D

    if ax is None:
        _, ax = plt.subplots(figsize=(11, 5))

    low_ids = {id(s) for s in low_limiters}
    high_ids = {id(s) for s in high_limiters}
    role_color = {'low': 'tab:blue', 'high': 'tab:orange', 'low+high': 'tab:green'}

    def role_of(ref):
        r = [name for name, ids in (('low', low_ids), ('high', high_ids)) if id(ref) in ids]
        return '+'.join(r) if r else None

    # context (non-limiter) references first, faint gray, underneath everything
    n_other = 0
    for i, ref in enumerate(reference_spectra):
        if role_of(ref) is None:
            ax.plot(valid_energies, A[:, i], color='0.45', alpha=0.5, linewidth=0.8, zorder=1)
            n_other += 1

    # limiting references, colored by role; all co-limiters share the role color
    role_counts = Counter()
    for i, ref in enumerate(reference_spectra):
        role = role_of(ref)
        if role is None:
            continue
        ax.plot(valid_energies, A[:, i], color=role_color[role], alpha=0.75, linewidth=1.3, zorder=3)
        role_counts[role] += 1

    # the full sample spectrum at ALL its energies: the portion outside the common
    # range (where no references are interpolated) is drawn dimmed and dashed so
    # every sample energy is visible, ...
    sample_energies = sample_spectrum.data_df.index.values
    sample_norm = sample_spectrum.data_df['norm'].values
    faint_sample, = ax.plot(sample_energies, sample_norm, color='black', alpha=0.5,
                            linewidth=1, linestyle='--', zorder=2)
    # ... while the in-range response vector b is drawn bold and solid on top.
    sample_line, = ax.plot(valid_energies, b, color='black', linewidth=2, zorder=4)
    handles = [sample_line, faint_sample]
    labels = [f'{sample_spectrum.file_name} (sample, in range)',
              'sample (all energies)']

    # one legend row per non-empty limiter role (names live on the bound lines below)
    for role in ('low', 'high', 'low+high'):
        n = role_counts.get(role, 0)
        if n:
            handles.append(Line2D([], [], color=role_color[role], alpha=0.75, linewidth=1.3))
            labels.append(f'{role} limiter{"" if n == 1 else "s"} ({n})')

    # one proxy entry stands in for every faded reference
    if n_other:
        handles.append(Line2D([], [], color='0.45', alpha=0.7, linewidth=0.8))
        labels.append(f'other references ({n_other})')

    # exact common-range bounds (red dashed) and the sampled-energy extent (gray dotted).
    # every low/high limiter shares the limiting energy, so read it off the first one;
    # summarise the limiter names so a big tie does not blow out the legend width.
    def summarize(spectra):
        names = [s.file_name for s in spectra]
        return ', '.join(names) if len(names) <= 2 else f'{len(names)} spectra'

    energy_min = low_limiters[0].data_df.index.values[0]
    energy_max = high_limiters[0].data_df.index.values[-1]
    lo = ax.axvline(energy_min, color='tab:red', linestyle='--', linewidth=1, zorder=2)
    hi = ax.axvline(energy_max, color='tab:red', linestyle='--', linewidth=1, zorder=2)
    ax.axvline(valid_energies[0], color='gray', linestyle=':', linewidth=1, zorder=2)
    ax.axvline(valid_energies[-1], color='gray', linestyle=':', linewidth=1, zorder=2)
    handles += [lo, hi, Line2D([], [], color='gray', linestyle=':', linewidth=1)]
    labels += [
        f'low bound {energy_min:.1f} eV (limited by {summarize(low_limiters)})',
        f'high bound {energy_max:.1f} eV (limited by {summarize(high_limiters)})',
        f'sampled extent [{valid_energies[0]:.1f}, {valid_energies[-1]:.1f}] eV',
    ]

    ax.set_xlabel('Energy (eV)')
    ax.set_ylabel('Normalized Fluorescence')
    ax.set_title(f'Interpolated references vs {sample_spectrum.file_name} '
                 f'({len(valid_energies)} energies, {len(reference_spectra)} references)')
    # legend outside the axes on the right so it never covers the data
    ax.legend(handles, labels, loc='upper left', bbox_to_anchor=(1.02, 1),
              fontsize='small', borderaxespad=0.0)
    return ax

In [ ]:
# Demonstrate on the three references used elsewhere in the notebook ...
valid_energies, A, b, low_limiters, high_limiters = \
    interpolate_references_at_sample_energies(reference_spectra, unknown_spectrum)
fig, ax = plt.subplots(figsize=(11, 5))
plot_interpolated_references(
    valid_energies, A, b, low_limiters, high_limiters,
    reference_spectra, unknown_spectrum, ax=ax,
)
plt.show()

# ... and on the full reference pool, to confirm it stays readable with many
# references (only the sample and the limiting spectra are highlighted; the rest
# recede to faint gray and collapse into a single legend entry).
ve, A_all, b_all, low_all, high_all = \
    interpolate_references_at_sample_energies(reference_spectra_list, unknown_spectrum)
fig, ax = plt.subplots(figsize=(11, 5))
plot_interpolated_references(
    ve, A_all, b_all, low_all, high_all,
    reference_spectra_list, unknown_spectrum, ax=ax,
)
plt.show()

In [ ]:
energies3, A3, b3, *_ = interpolate_references_at_sample_energies(reference_spectra, unknown_spectrum)

In [ ]:
def plot_spectrum_fit(energies, b, fitted, residuals, ax):
    ax.plot(energies, b, label='unknown', color='black')
    ax.plot(energies, fitted, label='fit', color='red', linestyle='--')
    ax.scatter(energies, residuals, color='orange', label='residuals', marker='o', s=10.0)
    ax.axhline(0, color='black', linewidth=0.5)
    ax.set_xlabel('Energy (eV)')
    ax.set_ylabel('Normalized Fluorescence')
    ax.legend()

nnls_coef, nnls_fitted, nnls_residuals = fit_nnls(A3, b3)

print('=== NNLS ===')
for ref_coef, ref_name in sorted(zip(nnls_coef, ref_names), reverse=True):
    print(f"  {ref_coef:.6f}: {ref_name}")
nnls_rmse = np.sqrt(np.mean(np.square(nnls_residuals)))
print(f'RMSE: {nnls_rmse:.6f}')

olstats_coef, olstats_fitted, olstats_residuals, olstats_result = fit_ols_with_statistics(A3, b3)

print('\n=== OLS with statistics ===')
for ref_coef, ref_name in sorted(zip(olstats_coef, ref_names), reverse=True):
    print(f"  {ref_coef:.6f}: {ref_name}")
olstats_rmse = np.sqrt(np.mean(np.square(olstats_residuals)))
print(f'RMSE: {olstats_rmse:.6f}')
print(olstats_result.summary())

ols_manual_coef, ols_manual_fitted, ols_manual_residuals = fit_ols(A3, b3)

print('\n=== OLS (manual) ===')
for ref_coef, ref_name in sorted(zip(ols_manual_coef, ref_names), reverse=True):
    print(f"  {ref_coef:.6f}: {ref_name}")
ols_manual_rmse = np.sqrt(np.mean(np.square(ols_manual_residuals)))
print(f'RMSE: {ols_manual_rmse:.6f}')

fig, (ax_nnls, ax_ols, ax_ols_manual) = plt.subplots(3, 1, figsize=(10, 12))
plot_spectrum_fit(energies3, b3, nnls_fitted, nnls_residuals, ax=ax_nnls)
ax_nnls.set_title(f'NNLS Fit: {unknown_spectrum.file_name}')
plot_spectrum_fit(energies3, b3, olstats_fitted, olstats_residuals, ax=ax_ols)
ax_ols.set_title(f'OLS Fit: {unknown_spectrum.file_name}')
plot_spectrum_fit(energies3, b3, ols_manual_fitted, ols_manual_residuals, ax=ax_ols_manual)
ax_ols_manual.set_title(f'OLS (manual) Fit: {unknown_spectrum.file_name}')
plt.tight_layout()
plt.show()


In [ ]:
def calculate_acf(residuals):
    n = len(residuals)
    n_lags = min(40, n // 2)
    x = residuals - residuals.mean()
    var = np.dot(x, x) / n
    acf_values = np.array(
        [1.0] + [np.dot(x[:-lag], x[lag:]) / (n * var) for lag in range(1, n_lags + 1)]
    )
    lags = np.arange(n_lags + 1)
    return lags, acf_values


In [ ]:
lags, acf_values = calculate_acf(nnls_residuals)

print(f'ACF at lag 0: {acf_values[0]:.4f}')
print(f'ACF at lag 1: {acf_values[1]:.4f}')
print(f'ACF at lag 2: {acf_values[2]:.4f}')
print(f'ACF at lag 5: {acf_values[5]:.4f}')


In [ ]:
def plot_residuals_histogram(residuals, ax, bins=15):
    mean = residuals.mean()
    std = residuals.std()
    ax.hist(residuals, bins=bins, color="orange", alpha=0.7, edgecolor="white")
    ax.axvline(mean, color="red", linestyle="--", label=f"mean={mean:.4f}")
    ax.axvline(mean + std, color="steelblue", linestyle=":", label=f"+1 std={mean + std:.4f}")
    ax.axvline(mean - std, color="steelblue", linestyle=":", label=f"-1 std={mean - std:.4f}")
    ax.axvline(0, color="black", linewidth=0.8)
    ax.set_xlabel("Residual")
    ax.set_ylabel("Count")
    ax.yaxis.set_major_locator(plt.MaxNLocator(integer=True))
    ax.legend()

fig, ax = plt.subplots(figsize=(10, 4))
plot_residuals_histogram(nnls_residuals, ax=ax)
ax.set_title('Residuals (NNLS)')
plt.tight_layout()
plt.show()

In [ ]:
def plot_acf(lags, acf_values, n, ax):
    ci_95 = 1.96 / np.sqrt(n)
    ax.bar(lags, acf_values, color='orange', alpha=0.7)
    ax.axhline(ci_95, color='red', linestyle='--', label='95% CI (white noise)')
    ax.axhline(-ci_95, color='red', linestyle='--')
    ax.axhline(0, color='black', linewidth=0.5)
    ax.set_xlabel('Lag')
    ax.set_ylabel('Autocorrelation')
    ax.legend()


fig, ax = plt.subplots(figsize=(10, 4))
plot_acf(lags, acf_values, len(nnls_residuals), ax=ax)
ax.set_title('Autocorrelation Function of Residuals')
plt.tight_layout()
plt.show()

In [ ]:
def moving_block_holdout_bootstrap(A, b, fitted, residuals, rng, n_bootstrap=1000):
    """ Early development version """
    n = len(residuals)
    block_length = max(1, int(np.round(n ** (1 / 3))))
    n_blocks_needed = int(np.ceil(n / block_length))
    n_full_blocks = n // block_length
    n_holdout_blocks = round(n_full_blocks / 3)
    # assume n=198 and block_length=6
    # then block_starts looks like
    #  array([  0,   1,   2, ..., 192 ])
    block_starts = np.arange(n - block_length + 1)

    bootstrap_coefs = np.zeros((n_bootstrap, A.shape[1]))
    bootstrap_pes = np.zeros(n_bootstrap)
    holdout_masks = np.zeros((n_bootstrap, n), dtype=bool)

    for i in range(n_bootstrap):
        # Randomly select non-contiguous holdout blocks totaling ~1/3 of the data
        # holdout_block_indices look like
        #  array([ 6, 20,  8,  3, 16, 28, 24,  2, 32, 22, 18])
        holdout_block_indices = rng.choice(n_full_blocks, size=n_holdout_blocks, replace=False)
        holdout_mask = np.zeros(n, dtype=bool)
        for idx in holdout_block_indices:
            holdout_mask[idx * block_length:(idx + 1) * block_length] = True
        holdout_masks[i] = holdout_mask
        train_mask = ~holdout_mask

        # Restrict bootstrap block starts to positions that don't overlap any holdout block
        valid_block_starts = block_starts[
            np.array([not holdout_mask[s:s + block_length].any() for s in block_starts])
        ]

        # Build bootstrap sample from moving blocks of residuals (holdout positions excluded)
        sampled_starts = rng.choice(valid_block_starts, size=n_blocks_needed, replace=True)
        bootstrap_residuals = np.concatenate(
            [residuals[s:s + block_length] for s in sampled_starts]
        )[:n]
        bootstrap_b = fitted + bootstrap_residuals

        # Fit on bootstrap data with holdout blocks removed
        bootstrap_coef, _ = scipy.optimize.nnls(A[train_mask], bootstrap_b[train_mask])
        bootstrap_coefs[i] = bootstrap_coef

        # Prediction error on real data at the held-out positions
        holdout_residuals = A[holdout_mask] @ bootstrap_coef - b[holdout_mask]
        bootstrap_pes[i] = np.sqrt(np.mean(np.square(holdout_residuals)))

    return bootstrap_coefs, bootstrap_pes, block_length, n_holdout_blocks, holdout_masks


rng = np.random.default_rng(seed=42)
n_bootstrap = 1000

bootstrap_coefs, bootstrap_pes, block_length, n_holdout_blocks, holdout_masks = \
    moving_block_holdout_bootstrap(A3, b3, nnls_fitted, nnls_residuals, rng, n_bootstrap=n_bootstrap)

n = len(nnls_residuals)
print(f'n={n}, block_length={block_length}, n_holdout_blocks={n_holdout_blocks} (~{n_holdout_blocks * block_length / n:.0%} of data)')
print(f'Coefficient means: {bootstrap_coefs.mean(axis=0)}')
print(f'Coefficient stds:  {bootstrap_coefs.std(axis=0)}')
print(f'Prediction error mean={bootstrap_pes.mean():.6f}, std={bootstrap_pes.std():.6f}')

In [ ]:
def select_holdout_blocks(n, rng, n_bootstrap=1000):
    """Pre-compute holdout masks and bootstrap block samples for n_bootstrap iterations.

    Captures all randomness so the same draws can be reused across multiple
    reference subsets via do_moving_block_holdout_bootstrap.

    Returns
    -------
    holdout_masks   : (n_bootstrap, n) bool array
    sampled_starts  : (n_bootstrap, n_blocks_needed) int array — block start indices for resampling
    block_length    : int
    n_holdout_blocks : int
    """
    block_length = max(1, int(np.round(n ** (1 / 3))))
    n_blocks_needed = int(np.ceil(n / block_length))
    n_full_blocks = n // block_length
    n_holdout_blocks = round(n_full_blocks / 3)
    block_starts = np.arange(n - block_length + 1)

    print(f'n={n}, block_length={block_length}, n_full_blocks={n_full_blocks}')
    print(f'n_holdout_blocks={n_holdout_blocks} (~{n_holdout_blocks * block_length / n:.0%} of data), '
          f'n_blocks_needed={n_blocks_needed}')

    holdout_masks = np.zeros((n_bootstrap, n), dtype=bool)
    sampled_starts = np.zeros((n_bootstrap, n_blocks_needed), dtype=int)

    for i in range(n_bootstrap):
        holdout_block_indices = rng.choice(n_full_blocks, size=n_holdout_blocks, replace=False)
        holdout_mask = np.zeros(n, dtype=bool)
        for idx in holdout_block_indices:
            holdout_mask[idx * block_length:(idx + 1) * block_length] = True
        holdout_masks[i] = holdout_mask

        valid_block_starts = block_starts[
            np.array([not holdout_mask[s:s + block_length].any() for s in block_starts])
        ]
        sampled_starts[i] = rng.choice(valid_block_starts, size=n_blocks_needed, replace=True)

    return holdout_masks, sampled_starts, block_length, n_holdout_blocks


def do_moving_block_holdout_bootstrap(A, b, fitted, residuals, holdout_masks, sampled_starts, block_length):
    """Compute holdout prediction error for each pre-drawn bootstrap iteration.

    Parameters
    ----------
    A               : (n, n_refs) design matrix for one reference subset
    b               : (n,) observed spectrum values
    fitted          : (n,) full-data fitted values
    residuals       : (n,) full-data residuals (fitted - b)
    holdout_masks   : (n_bootstrap, n) bool array from select_holdout_blocks
    sampled_starts  : (n_bootstrap, n_blocks_needed) int array from select_holdout_blocks
    block_length    : int from select_holdout_blocks

    Returns
    -------
    bootstrap_coefs : (n_bootstrap, n_refs) float array
    bootstrap_pes   : (n_bootstrap,) float array — per-iteration holdout RMSE
    """
    n = len(b)
    n_bootstrap = len(holdout_masks)
    bootstrap_coefs = np.zeros((n_bootstrap, A.shape[1]))
    # Instead of computing each iteration's holdout RMSE inline with
    # np.sqrt(np.mean(np.square(...))) (a np.square/np.mean/np.sqrt trio per iteration,
    # ~2.32M times over the notebook's workload), accumulate each iteration's holdout
    # sum-of-squares and held-out point count as scalars and take the RMSE for all
    # iterations at once after the loop. This moves the mean/sqrt reduction out of the
    # hot loop (~10-15% faster here) and, because it stores scalars rather than a
    # fixed-width residual array, stays correct when the number of held-out points
    # varies between iterations (e.g. select_holdout_blocks_v4/v5).
    holdout_sum_of_squares = np.zeros(n_bootstrap)
    holdout_point_counts = np.zeros(n_bootstrap)

    # Vectorized replacement for the per-iteration block gather this loop used to do. The
    # original code, inside `for i in range(n_bootstrap):`, called ~2.32M times total across
    # 2324 reference combinations x 1000 bootstrap iterations, was:
    #
    #     bootstrap_residuals = np.concatenate(
    #         [residuals[s:s + block_length] for s in sampled_starts[i]]
    #     )[:n]
    #     bootstrap_b = fitted + bootstrap_residuals
    #
    # sampled_starts (shape (n_bootstrap, n_blocks_needed)) is fully known before the loop
    # starts, so instead of re-gathering blocks one bootstrap iteration at a time, gather all
    # of them at once with a single fancy-indexing operation:
    #
    #   1. block_offsets = [0, 1, ..., block_length - 1].
    #   2. Broadcasting sampled_starts[:, :, None] (n_bootstrap, n_blocks_needed, 1) against
    #      block_offsets[None, None, :] (1, 1, block_length) gives block_indices of shape
    #      (n_bootstrap, n_blocks_needed, block_length), where block_indices[i, j] is the
    #      block_length run of consecutive residual indices for bootstrap i's j-th sampled
    #      block (i.e. sampled_starts[i, j] + block_offsets).
    #   3. residuals[block_indices] gathers those residual values, same shape.
    #   4. .reshape(n_bootstrap, -1) flattens the (n_blocks_needed, block_length) axes into a
    #      single axis per bootstrap row, in the same block-by-block, then-within-block order
    #      that np.concatenate produced per iteration in the original code.
    #   5. [:, :n] truncates each row to n, matching the original per-iteration `[:n]` (the
    #      concatenated blocks run past n since n_blocks_needed * block_length >= n).
    #
    # all_bootstrap_residuals ends up with shape (n_bootstrap, n): row i is exactly what the
    # old code computed as bootstrap_residuals for iteration i. Moving the gather out of the
    # hot loop cut ~27% off this function's time during profiling.
    block_offsets = np.arange(block_length)
    block_indices = sampled_starts[:, :, None] + block_offsets[None, None, :]
    all_bootstrap_residuals = residuals[block_indices].reshape(n_bootstrap, -1)[:, :n]

    for i in range(n_bootstrap):
        holdout_mask = holdout_masks[i]
        train_mask = ~holdout_mask

        bootstrap_b = fitted + all_bootstrap_residuals[i]

        bootstrap_coef, _ = scipy.optimize.nnls(A[train_mask], bootstrap_b[train_mask])
        bootstrap_coefs[i] = bootstrap_coef

        holdout_residuals = A[holdout_mask] @ bootstrap_coef - b[holdout_mask]
        holdout_sum_of_squares[i] = holdout_residuals @ holdout_residuals
        holdout_point_counts[i] = holdout_residuals.shape[0]

    # per-iteration holdout RMSE, reduced for all iterations at once
    bootstrap_pes = np.sqrt(holdout_sum_of_squares / holdout_point_counts)

    return bootstrap_coefs, bootstrap_pes


In [ ]:
def select_holdout_blocks_v2(n, rng, n_bootstrap=1000):
    """Like select_holdout_blocks, but shifts the holdout block grid by a random
    offset in [0, block_length) each iteration so every position has roughly
    equal probability of being held out rather than only positions aligned to
    multiples of block_length.

    Returns
    -------
    holdout_masks    : (n_bootstrap, n) bool array
    sampled_starts   : (n_bootstrap, n_blocks_needed) int array
    block_length     : int
    n_holdout_blocks : int — representative value; actual count may vary ±1 per
                       iteration depending on how many shifted blocks fit in [0, n)
    """
    block_length = max(1, int(np.round(n ** (1 / 3))))
    n_blocks_needed = int(np.ceil(n / block_length))
    n_full_blocks = n // block_length
    n_holdout_blocks = round(n_full_blocks / 3)
    block_starts = np.arange(n - block_length + 1)

    print(f'n={n}, block_length={block_length}, n_full_blocks={n_full_blocks}')
    print(f'n_holdout_blocks={n_holdout_blocks} (~{n_holdout_blocks * block_length / n:.0%} of data), '
          f'n_blocks_needed={n_blocks_needed}')

    holdout_masks = np.zeros((n_bootstrap, n), dtype=bool)
    sampled_starts = np.zeros((n_bootstrap, n_blocks_needed), dtype=int)

    for i in range(n_bootstrap):
        offset = int(rng.integers(0, block_length))
        shifted_grid = np.arange(offset, n - block_length + 1, block_length)
        n_shifted = len(shifted_grid)
        n_shifted_holdout = round(n_shifted / 3)

        holdout_block_indices = rng.choice(n_shifted, size=n_shifted_holdout, replace=False)
        holdout_mask = np.zeros(n, dtype=bool)
        for idx in holdout_block_indices:
            start = shifted_grid[idx]
            holdout_mask[start:start + block_length] = True
        holdout_masks[i] = holdout_mask

        valid_block_starts = block_starts[
            np.array([not holdout_mask[s:s + block_length].any() for s in block_starts])
        ]
        sampled_starts[i] = rng.choice(valid_block_starts, size=n_blocks_needed, replace=True)

    return holdout_masks, sampled_starts, block_length, n_holdout_blocks


In [ ]:
def select_holdout_blocks_v3(n, rng, n_bootstrap=1000):
    """Like select_holdout_blocks_v2 (random per-iteration shift) but uses a circular
    (modular) block grid with exactly n_full_blocks blocks, so every position is in
    exactly one block regardless of offset. This eliminates the v2 boundary effect.

    For offsets > 0, one block wraps around the end of the array, mixing high-index
    and low-index (high-energy and low-energy) positions. That is the cost of
    achieving uniform coverage.

    Returns
    -------
    holdout_masks    : (n_bootstrap, n) bool array
    sampled_starts   : (n_bootstrap, n_blocks_needed) int array
    block_length     : int
    n_holdout_blocks : int
    """
    block_length = max(1, int(np.round(n ** (1 / 3))))
    n_blocks_needed = int(np.ceil(n / block_length))
    n_full_blocks = n // block_length
    n_holdout_blocks = round(n_full_blocks / 3)
    resample_block_starts = np.arange(n - block_length + 1)

    print(f'n={n}, block_length={block_length}, n_full_blocks={n_full_blocks}')
    print(f'n_holdout_blocks={n_holdout_blocks} (~{n_holdout_blocks * block_length / n:.0%} of data), '
          f'n_blocks_needed={n_blocks_needed}')

    holdout_masks = np.zeros((n_bootstrap, n), dtype=bool)
    sampled_starts = np.zeros((n_bootstrap, n_blocks_needed), dtype=int)

    for i in range(n_bootstrap):
        offset = int(rng.integers(0, block_length))
        # n_full_blocks evenly-spaced blocks, wrapping circularly at n
        circular_starts = (np.arange(n_full_blocks) * block_length + offset) % n

        holdout_block_indices = rng.choice(n_full_blocks, size=n_holdout_blocks, replace=False)
        holdout_mask = np.zeros(n, dtype=bool)
        for idx in holdout_block_indices:
            start = int(circular_starts[idx])
            for j in range(block_length):
                holdout_mask[(start + j) % n] = True
        holdout_masks[i] = holdout_mask

        valid_block_starts = resample_block_starts[
            np.array([not holdout_mask[s:s + block_length].any() for s in resample_block_starts])
        ]
        sampled_starts[i] = rng.choice(valid_block_starts, size=n_blocks_needed, replace=True)

    return holdout_masks, sampled_starts, block_length, n_holdout_blocks


In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))

ax.plot(energies3, b3, label='unknown spectrum', color='black', linewidth=1.0)
ax.scatter(
    energies3[~holdout_masks[-1]], b3[~holdout_masks[-1]],
    color='steelblue', s=10, zorder=4, alpha=0.6,
    label=f'training ({(~holdout_masks[-1]).sum()} points)',
)
ax.scatter(
    energies3[holdout_masks[-1]], b3[holdout_masks[-1]],
    color='orange', s=20, zorder=5,
    label=f'holdout ({holdout_masks[-1].sum()} points)',
)

ax.set_xlabel('Energy (eV)')
ax.set_ylabel('Normalized Fluorescence')
ax.set_title(
    f'Last Holdout Mask — {unknown_spectrum.file_name}\n'
    f'block_length={block_length}, n_holdout_blocks={n_holdout_blocks}'
)
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
def plot_bootstrap_summary(
    energies, b, fitted, residuals,
    lags, acf_values,
    bootstrap_coefs, bootstrap_pes,
    coef, ref_names,
    spectrum_name, n_bootstrap,
    title_prefix=None,
):
    import matplotlib.gridspec as gridspec

    n = len(residuals)
    n_refs = len(ref_names)
    n_cols = n_refs + 1
    rmse = np.sqrt(np.mean(residuals ** 2))

    fig = plt.figure(figsize=(4 * n_cols, 16))
    gs = gridspec.GridSpec(4, n_cols, figure=fig)

    ax_fit = fig.add_subplot(gs[0, :])
    ax_acf = fig.add_subplot(gs[1, :n_cols // 2])
    ax_resid_hist = fig.add_subplot(gs[1, n_cols // 2:])
    axs = np.array([[fig.add_subplot(gs[row, col]) for col in range(n_cols)] for row in range(2, 4)])

    plot_spectrum_fit(energies, b, fitted, residuals, ax=ax_fit)
    ax_fit.set_title(spectrum_name)

    plot_acf(lags, acf_values, n, ax=ax_acf)
    ax_acf.set_title('Residual Autocorrelation Function')
    ax_acf.legend(fontsize=12)

    plot_residuals_histogram(residuals, ax=ax_resid_hist)
    ax_resid_hist.set_title('Residuals Histogram')
    ax_resid_hist.legend(fontsize=12)

    for j, (coef_mean, coef_i, name) \
            in enumerate(sorted(zip(bootstrap_coefs.mean(axis=0), range(n_refs), ref_names), reverse=True)):

        p2_5 = np.percentile(bootstrap_coefs[:, coef_i], 2.5)
        p97_5 = np.percentile(bootstrap_coefs[:, coef_i], 97.5)

        axs[0, j].hist(bootstrap_coefs[:, coef_i], bins=40, color='steelblue', alpha=0.7, edgecolor='white')
        axs[0, j].axvline(coef[coef_i], color='red', linestyle='--', label=f'observed={coef[coef_i]:.3f}')
        axs[0, j].axvline(coef_mean, color='blue', linestyle='--', label=f'mean={coef_mean:.3f}')
        axs[0, j].axvline(p2_5, color='green', linestyle=':', label=f'2.5%={p2_5:.3f}')
        axs[0, j].axvline(p97_5, color='green', linestyle=':', label=f'97.5%={p97_5:.3f}')
        axs[0, j].set_title(name, fontsize=9)
        axs[0, j].set_xlabel('Coefficient')
        axs[0, j].legend(fontsize=8)

        axs[1, j].violinplot(bootstrap_coefs[:, coef_i])
        axs[1, j].scatter(
            [0.95], [coef[coef_i]], color='red', zorder=5, marker='o', s=60,
            edgecolors='black', linewidths=0.8, label=f'observed={coef[coef_i]:.3f}',
        )
        axs[1, j].scatter(
            [1.05], [coef_mean], color='blue', zorder=5, marker='D', s=60,
            edgecolors='black', linewidths=0.8, label=f'mean={coef_mean:.3f}',
        )
        axs[1, j].set_title(name, fontsize=9)
        axs[1, j].legend(fontsize=8)

    pes_mean = bootstrap_pes.mean()
    pes_p2_5 = np.percentile(bootstrap_pes, 2.5)
    pes_p97_5 = np.percentile(bootstrap_pes, 97.5)

    axs[0, n_refs].hist(bootstrap_pes, bins=40, color='darkorange', alpha=0.7, edgecolor='white')
    axs[0, n_refs].axvline(rmse, color='red', linestyle='--', label=f'RMSE={rmse:.4f}')
    axs[0, n_refs].axvline(pes_mean, color='blue', linestyle='--', label=f'mean={pes_mean:.4f}')
    axs[0, n_refs].axvline(pes_p2_5, color='green', linestyle=':', label=f'2.5%={pes_p2_5:.4f}')
    axs[0, n_refs].axvline(pes_p97_5, color='green', linestyle=':', label=f'97.5%={pes_p97_5:.4f}')
    axs[0, n_refs].set_title('Holdout Prediction Error')
    axs[0, n_refs].set_xlabel('Prediction Error (RMSE)')
    axs[0, n_refs].legend(fontsize=8)

    parts = axs[1, n_refs].violinplot(bootstrap_pes)
    for pc in parts['bodies']:
        pc.set_facecolor('darkorange')
        pc.set_alpha(0.7)
    axs[1, n_refs].scatter(
        [0.95], [rmse], color='red', zorder=5, marker='o', s=60,
        edgecolors='black', linewidths=0.8, label=f'RMSE={rmse:.4f}',
    )
    axs[1, n_refs].scatter(
        [1.05], [pes_mean], color='blue', zorder=5, marker='D', s=60,
        edgecolors='black', linewidths=0.8, label=f'mean={pes_mean:.4f}',
    )
    axs[1, n_refs].set_title('Holdout Prediction Error')
    axs[1, n_refs].legend(fontsize=8)

    coef_violin_axes = axs[1, :n_refs]
    all_ylims = [ax.get_ylim() for ax in coef_violin_axes]
    global_ymin = min(lo for lo, hi in all_ylims)
    global_ymax = max(hi for lo, hi in all_ylims)
    for ax in coef_violin_axes:
        ax.set_ylim(global_ymin, global_ymax)

    suptitle = f'Moving Block Holdout Bootstrap Distributions ({n_bootstrap} iterations)'
    if title_prefix is not None:
        suptitle = f'{title_prefix} — {suptitle}'
    plt.suptitle(suptitle, fontsize=13)
    # tight_layout doesn't account for suptitle; top=0.97 reserves space so it doesn't overlap ax_fit
    plt.tight_layout(rect=[0, 0, 1, 0.97])
    plt.show()

In [ ]:
plot_bootstrap_summary(
    energies3, b3, nnls_fitted, nnls_residuals,
    lags, acf_values,
    bootstrap_coefs, bootstrap_pes,
    nnls_coef, ref_names,
    spectrum_name=unknown_spectrum.file_name,
    n_bootstrap=n_bootstrap,
)

In [ ]:
energies, A, b, *_ = interpolate_references_at_sample_energies(reference_spectra, unknown_spectrum)

In [ ]:
def select_holdout_blocks_v4(n, rng, n_bootstrap=1000, block_length_min=6, block_length_max=10):
    """Like select_holdout_blocks but partitions the data into blocks of random
    lengths drawn uniformly from [block_length_min, block_length_max] each iteration,
    so holdout boundaries fall at different positions every iteration.

    Bootstrap resampling uses fixed length block_length_min to stay compatible
    with do_moving_block_holdout_bootstrap.

    Returns
    -------
    holdout_masks    : (n_bootstrap, n) bool array
    sampled_starts   : (n_bootstrap, n_blocks_needed) int array
    block_length     : int — block_length_min, used for resampling
    n_holdout_blocks : int — mean holdout block count across iterations (rounded)
    """
    n_blocks_needed = int(np.ceil(n / block_length_min))
    resample_block_starts = np.arange(n - block_length_min + 1)

    holdout_masks = np.zeros((n_bootstrap, n), dtype=bool)
    sampled_starts = np.zeros((n_bootstrap, n_blocks_needed), dtype=int)
    total_holdout_blocks = 0

    for i in range(n_bootstrap):
        # Partition [0, n) into blocks of random lengths
        block_starts_i, block_lengths_i = [], []
        pos = 0
        while pos < n:
            bl = int(rng.integers(block_length_min, block_length_max + 1))
            bl = min(bl, n - pos)
            block_starts_i.append(pos)
            block_lengths_i.append(bl)
            pos += bl

        n_blocks_i = len(block_lengths_i)
        n_holdout_i = round(n_blocks_i / 3)
        total_holdout_blocks += n_holdout_i

        holdout_block_indices = rng.choice(n_blocks_i, size=n_holdout_i, replace=False)
        holdout_mask = np.zeros(n, dtype=bool)
        for idx in holdout_block_indices:
            s = block_starts_i[idx]
            holdout_mask[s:s + block_lengths_i[idx]] = True
        holdout_masks[i] = holdout_mask

        valid_resample_starts = resample_block_starts[
            np.array([not holdout_mask[s:s + block_length_min].any() for s in resample_block_starts])
        ]
        sampled_starts[i] = rng.choice(valid_resample_starts, size=n_blocks_needed, replace=True)

    n_holdout_blocks = round(total_holdout_blocks / n_bootstrap)
    mean_holdout_frac = total_holdout_blocks * (block_length_min + block_length_max) / 2 / (n_bootstrap * n)
    print(f'n={n}, block_length=[{block_length_min}, {block_length_max}], n_blocks_needed={n_blocks_needed}')
    print(f'avg n_holdout_blocks={n_holdout_blocks} (~{mean_holdout_frac:.0%} of data)')

    return holdout_masks, sampled_starts, block_length_min, n_holdout_blocks


In [ ]:
def select_holdout_blocks_v5(n, rng, n_bootstrap=1000, block_length_min=6, block_length_max=10):
    """Like select_holdout_blocks_v4 but reverses every other holdout mask, so the
    truncated final block alternates between the high-energy and low-energy end of
    the spectrum rather than always falling at the high-energy end.
    """
    n_blocks_needed = int(np.ceil(n / block_length_min))
    resample_block_starts = np.arange(n - block_length_min + 1)

    holdout_masks = np.zeros((n_bootstrap, n), dtype=bool)
    sampled_starts = np.zeros((n_bootstrap, n_blocks_needed), dtype=int)
    total_holdout_blocks = 0

    for i in range(n_bootstrap):
        block_starts_i, block_lengths_i = [], []
        pos = 0
        while pos < n:
            bl = int(rng.integers(block_length_min, block_length_max + 1))
            bl = min(bl, n - pos)
            block_starts_i.append(pos)
            block_lengths_i.append(bl)
            pos += bl

        n_blocks_i = len(block_lengths_i)
        n_holdout_i = round(n_blocks_i / 3)
        total_holdout_blocks += n_holdout_i

        holdout_block_indices = rng.choice(n_blocks_i, size=n_holdout_i, replace=False)
        holdout_mask = np.zeros(n, dtype=bool)
        for idx in holdout_block_indices:
            s = block_starts_i[idx]
            holdout_mask[s:s + block_lengths_i[idx]] = True

        if i % 2 == 1:
            holdout_mask = holdout_mask[::-1]

        holdout_masks[i] = holdout_mask

        valid_resample_starts = resample_block_starts[
            np.array([not holdout_mask[s:s + block_length_min].any() for s in resample_block_starts])
        ]
        sampled_starts[i] = rng.choice(valid_resample_starts, size=n_blocks_needed, replace=True)

    n_holdout_blocks = round(total_holdout_blocks / n_bootstrap)
    mean_holdout_frac = total_holdout_blocks * (block_length_min + block_length_max) / 2 / (n_bootstrap * n)
    print(f'n={n}, block_length=[{block_length_min}, {block_length_max}], n_blocks_needed={n_blocks_needed}')
    print(f'avg n_holdout_blocks={n_holdout_blocks} (~{mean_holdout_frac:.0%} of data)')

    return holdout_masks, sampled_starts, block_length_min, n_holdout_blocks


### Visualizing holdout block structure

The write-up below argues about *where holdout block boundaries fall* — whether the
grid is aligned or shifted, whether a block wraps around the end of the array, whether
the truncated final block always lands at the high-energy end. Those are claims about
the masks themselves, so plot the masks: `plot_holdout_block_structure` shows the
geometry each version produces, and the per-version holdout frequency std it reports
is the same statistic the results table below is built from.

This only needs the selector output — no NNLS and no `do_moving_block_holdout_bootstrap`
— so it runs in seconds, unlike the prediction error comparison further down. The
`n=...`, `block_length=...` lines it prints come from the selectors themselves.

In [ ]:
def contiguous_run_lengths(holdout_masks):
    """Lengths of every contiguous run of held-out positions, pooled over iterations.

    Each row of holdout_masks is padded with False on both sides so runs touching an
    edge are still bounded by a transition; np.diff then marks run starts with +1 and
    run ends with -1. np.argwhere returns those in row-major order, so within a row
    the k-th start pairs with the k-th end and the difference of their column indices
    is the run length.

    Returns
    -------
    (n_runs,) int array -- one entry per run, pooled over all iterations
    """
    padded = np.zeros((holdout_masks.shape[0], holdout_masks.shape[1] + 2), dtype=np.int8)
    padded[:, 1:-1] = holdout_masks
    transitions = np.diff(padded, axis=1)
    run_starts = np.argwhere(transitions == 1)
    run_ends = np.argwhere(transitions == -1)
    return run_ends[:, 1] - run_starts[:, 1]


def smooth(values, window):
    """Centered moving average, with the window shrunk at the edges.

    Dividing a same-mode convolution of the values by the same convolution of an
    all-ones array normalizes each output point by however many input points actually
    contributed, so edge points are averaged over a partial window instead of being
    pulled toward zero by the implicit zero padding. That matters here because the
    edges are exactly where the versions differ.
    """
    kernel = np.ones(window)
    return (
        np.convolve(values, kernel, mode='same')
        / np.convolve(np.ones_like(values), kernel, mode='same')
    )

In [ ]:
def plot_holdout_block_structure(
    n, energies, selectors, n_bootstrap=1000,
    n_raster_iterations=25, raster_window=54, seed=0,
):
    """Compare the holdout mask *geometry* produced by select_holdout_blocks v1-v5.

    The v1-v5 write-up argues about where holdout block boundaries fall: whether the
    grid is aligned or shifted, whether a block wraps around the end of the array,
    whether the truncated final block always lands at the high-energy end. Those are
    all claims about the masks themselves, but the only geometric evidence elsewhere
    in this notebook is a single number per version (the holdout frequency std). This
    figure shows the masks directly, so the claims can be read off a plot.

    Only the selector output is needed -- no NNLS, no do_moving_block_holdout_bootstrap
    -- so this runs in seconds rather than the minutes the prediction error
    comparisons take.

    Parameters
    ----------
    n                   : int -- number of energy points (len(b))
    energies            : (n,) array -- energy axis, for the full-width marginal plots
    selectors           : list of (label, select_holdout_blocks_fn, color)
    n_bootstrap         : int -- iterations to draw from each selector
    n_raster_iterations : int -- iterations to show in the rasters (row 1)
    raster_window       : int -- positions to show at each end of the spectrum in the
                          rasters. All n positions will not resolve into visible cells
                          at this figure width, and the interesting differences between
                          versions are edge effects, so each raster row shows the
                          low-energy and high-energy ends rather than a squeezed whole.
    seed                : int -- each selector gets its own default_rng(seed), so the
                          versions are compared on the same random stream

    Layout: one column per selector, four rows.
      1. holdout mask rasters at both ends of the spectrum, with the column marginal
         -- the per-position holdout frequency over all n -- beneath them
      2. held-out fraction per iteration
      3. realized contiguous held-out run lengths
      4. coverage relative to uniform, for both the holdout draw and the bootstrap
         resample draw
    """
    import matplotlib.gridspec as gridspec
    from matplotlib.colors import LinearSegmentedColormap

    # Draw from every selector up front: rows 2-4 share x-limits across columns, which
    # can only be decided once all versions are known.
    drawn = []
    for label, fn, color in selectors:
        print(f'\n=== {label} ===')
        holdout_masks, sampled_starts, block_length, n_holdout_blocks = \
            fn(n, np.random.default_rng(seed=seed), n_bootstrap=n_bootstrap)

        # Holding out a block also removes it from the pool of blocks available to
        # resample from (the valid_block_starts filter every selector applies), but the
        # resampling side is never plotted anywhere else in this notebook. Express both
        # draws as coverage relative to uniform so they share a single axis: 1.0 means
        # "drawn exactly as often as a perfectly even draw would". Smooth both over one
        # block length -- unsmoothed, the resample counts are dominated by Poisson noise
        # from the ~33k draws and the systematic structure is unreadable.
        frequency = holdout_masks.mean(axis=0)
        n_start_positions = n - block_length + 1
        resample_counts = np.bincount(
            sampled_starts.ravel(), minlength=n_start_positions,
        )[:n_start_positions].astype(float)

        drawn.append({
            'label': label,
            'color': color,
            'holdout_masks': holdout_masks,
            'block_length': block_length,
            'holdout_frequency': frequency,
            'holdout_fraction_per_iteration': holdout_masks.sum(axis=1) / n,
            'run_lengths': contiguous_run_lengths(holdout_masks),
            'n_start_positions': n_start_positions,
            'relative_holdout': smooth(frequency / frequency.mean(), block_length),
            'relative_resample': smooth(resample_counts / resample_counts.mean(), block_length),
        })

    fraction_min = min(d['holdout_fraction_per_iteration'].min() for d in drawn)
    fraction_max = max(d['holdout_fraction_per_iteration'].max() for d in drawn)
    # A handful of very long runs (several selected blocks landing adjacent) would
    # otherwise squeeze every informative bar into the first tenth of the axis, so
    # clip the shared x-limit to a high percentile and report the true max in-panel.
    run_length_limit = int(np.ceil(max(
        np.percentile(d['run_lengths'], 99.5) for d in drawn
    )))
    # One shared coverage scale across all five columns. Per-column autoscaling would
    # blow up each version's sampling noise to fill its own axis, making versions that
    # are in fact flat at 1.0 look as structured as v2, whose edges really do collapse.
    coverage_values = np.concatenate(
        [d['relative_holdout'] for d in drawn] + [d['relative_resample'] for d in drawn]
    )
    coverage_lo, coverage_hi = coverage_values.min(), coverage_values.max()
    coverage_pad = 0.05 * (coverage_hi - coverage_lo)

    n_raster = min(n_raster_iterations, n_bootstrap)
    window = min(raster_window, n // 2)

    fig = plt.figure(figsize=(5 * len(selectors), 16))
    outer = gridspec.GridSpec(
        4, len(selectors), figure=fig, height_ratios=[3, 1.3, 1.3, 1.3], hspace=0.45, wspace=0.25,
    )

    for j, d in enumerate(drawn):
        color = d['color']
        block_length = d['block_length']
        # White -> version color, so "held out" reads as ink on the page while hue
        # still carries version identity.
        cmap = LinearSegmentedColormap.from_list(f'holdout_{j}', ['white', color])

        # ------------------------------------------------------------------ row 1
        # Two rasters (low-energy end, high-energy end) over one full-width marginal:
        # the per-position holdout frequency, which is the column mean of the same
        # mask array the rasters are showing.
        inner = gridspec.GridSpecFromSubplotSpec(
            2, 2, subplot_spec=outer[0, j], height_ratios=[4, 1], hspace=0.30, wspace=0.10,
        )
        ax_low = fig.add_subplot(inner[0, 0])
        ax_high = fig.add_subplot(inner[0, 1], sharey=ax_low)
        ax_frequency = fig.add_subplot(inner[1, :])

        for ax, positions in (
            (ax_low, np.arange(window)),
            (ax_high, np.arange(n - window, n)),
        ):
            ax.imshow(
                d['holdout_masks'][:n_raster, positions],
                aspect='auto', interpolation='nearest', cmap=cmap, vmin=0, vmax=1,
                extent=[positions[0] - 0.5, positions[-1] + 0.5, n_raster, 0],
            )
            # Where an aligned grid would put its boundaries -- v1 sits exactly on
            # these, and the other versions' departure from them is the thing to see.
            for boundary in range(0, n, block_length):
                if positions[0] <= boundary <= positions[-1]:
                    ax.axvline(boundary - 0.5, color='0.45', linewidth=0.5, alpha=0.7)
            ax.set_xlabel('Position (index)', fontsize=8)
            ax.tick_params(labelsize=7)

        ax_low.set_ylabel('Bootstrap iteration', fontsize=8)
        ax_low.set_title('low-energy end', fontsize=8)
        ax_high.set_title('high-energy end', fontsize=8)
        ax_high.tick_params(labelleft=False)
        # Anchored to the left raster but centered over the pair, so the label does not
        # run into the neighbouring column.
        ax_low.annotate(
            f'{d["label"]}\nheld out, first {n_raster} iterations '
            f'(gray = aligned grid, spacing {block_length})',
            xy=(1.05, 1.22), xycoords='axes fraction', ha='center', va='bottom', fontsize=10,
        )

        frequency = d['holdout_frequency']
        ax_frequency.plot(energies, frequency, color=color, linewidth=0.9)
        ax_frequency.axhline(
            frequency.mean(), color='red', linestyle='--', linewidth=1.0,
            label=f'mean={frequency.mean():.3f}, std={frequency.std():.4f}',
        )
        ax_frequency.set_xlabel('Energy (eV)', fontsize=8)
        ax_frequency.set_ylabel('Fraction\nheld out', fontsize=8)
        ax_frequency.set_ylim(0, max(0.5, frequency.max() * 1.15))
        ax_frequency.tick_params(labelsize=7)
        ax_frequency.legend(fontsize=7, loc='lower center', framealpha=0.85)

        # ------------------------------------------------------------------ row 2
        # How much data each iteration holds out. v1-v3 hold out a constant count;
        # v4/v5 vary, which is the mechanism behind their wider prediction error CIs.
        ax_fraction = fig.add_subplot(outer[1, j])
        fractions = d['holdout_fraction_per_iteration']
        ax_fraction.hist(
            fractions, bins=30, range=(fraction_min, fraction_max), color=color, alpha=0.8,
        )
        ax_fraction.axvline(
            np.median(fractions), color='red', linestyle='--', linewidth=1.0,
            label=(
                f'median={np.median(fractions):.3f}\n'
                f'range=[{fractions.min():.3f}, {fractions.max():.3f}]'
            ),
        )
        ax_fraction.set_xlim(fraction_min - 0.01, fraction_max + 0.01)
        ax_fraction.set_xlabel('Fraction of points held out')
        ax_fraction.set_ylabel('Iterations')
        ax_fraction.set_title('Held-out fraction per iteration', fontsize=10)
        ax_fraction.legend(fontsize=7)

        # ------------------------------------------------------------------ row 3
        # Realized block geometry rather than the nominal block_length: adjacent
        # selected blocks merge into longer runs, and truncated or wrapped blocks show
        # up as short ones.
        ax_runs = fig.add_subplot(outer[2, j])
        run_lengths = d['run_lengths']
        ax_runs.hist(
            run_lengths, bins=np.arange(0.5, run_length_limit + 1.5), color=color, alpha=0.8,
        )
        ax_runs.axvline(
            block_length, color='red', linestyle='--', linewidth=1.0,
            label=(
                f'resample block_length={block_length}\n'
                f'median run={int(np.median(run_lengths))}, longest={run_lengths.max()}\n'
                f'runs beyond axis: {(run_lengths > run_length_limit).sum()}'
            ),
        )
        ax_runs.set_xlim(0, run_length_limit + 1)
        ax_runs.set_xlabel('Contiguous held-out run length (points)')
        ax_runs.set_ylabel('Runs')
        ax_runs.set_title('Realized held-out run lengths', fontsize=10)
        ax_runs.legend(fontsize=7)

        # ------------------------------------------------------------------ row 4
        # Holdout draw vs. resample draw, both relative to uniform (see the first loop).
        ax_coverage = fig.add_subplot(outer[3, j])
        n_start_positions = d['n_start_positions']

        ax_coverage.axhline(1.0, color='gray', linestyle=':', linewidth=1.0)
        ax_coverage.plot(
            energies, d['relative_holdout'], color=color, linewidth=1.4, label='held out',
        )
        ax_coverage.plot(
            energies[:n_start_positions], d['relative_resample'],
            color='dimgray', linewidth=1.4, linestyle='--', label='resample block start',
        )
        ax_coverage.set_ylim(coverage_lo - coverage_pad, coverage_hi + coverage_pad)
        ax_coverage.set_xlabel('Energy (eV)')
        ax_coverage.set_ylabel('Coverage relative\nto uniform', fontsize=8)
        ax_coverage.set_title(
            f'Holdout vs. resample coverage (smoothed over {block_length} points)', fontsize=10,
        )
        ax_coverage.legend(fontsize=7)

    fig.suptitle(
        f'select_holdout_blocks v1–v5 — holdout mask structure, '
        f'n={n}, {n_bootstrap} iterations, seed={seed}',
        fontsize=13, y=0.985,
    )
    # tight_layout cannot handle the nested GridSpecFromSubplotSpec in row 1, so the
    # spacing is set explicitly by the GridSpec above plus this margin adjustment.
    fig.subplots_adjust(left=0.04, right=0.99, top=0.90, bottom=0.04)
    plt.show()

In [ ]:
plot_holdout_block_structure(
    n=len(b),
    energies=energies,
    selectors=[
        ('v1 (aligned)',        select_holdout_blocks,    'steelblue'),
        ('v2 (shifted)',        select_holdout_blocks_v2, 'darkorange'),
        ('v3 (circular shift)', select_holdout_blocks_v3, 'crimson'),
        ('v4 (random lengths)', select_holdout_blocks_v4, 'mediumseagreen'),
        ('v5 (alternating)',    select_holdout_blocks_v5, 'mediumpurple'),
    ],
)

## Development of `select_holdout_blocks` v1–v5

### Background

The goal of `select_holdout_blocks` is to pre-draw holdout masks for a moving-block holdout bootstrap: each iteration holds out ~1/3 of the data in contiguous blocks, fits the model on the remaining 2/3, and measures prediction error on the held-out positions. Across 1000 iterations, the ideal behavior is that every position in the spectrum is held out with roughly equal frequency — otherwise, some energy regions contribute more than others to the aggregate prediction error estimate.

Two metrics were used to compare versions:
- **Holdout frequency std** across positions (lower = more uniform coverage)
- **PE distribution**: mean, median, and 95% CI of each, computed by resampling the 1000 bootstrap PE draws 5000 times and taking the 2.5th/97.5th percentiles

---

### v1 — Aligned grid (baseline)

The original version computes a fixed block length `L = round(n^(1/3))` (L=6 for n=198) and partitions the data into non-overlapping aligned blocks at positions 0, L, 2L, …. Each iteration randomly selects ~1/3 of these blocks for holdout. Because every position belongs to exactly one block and all blocks are selected with equal probability, holdout frequency is uniform in expectation.

---

### v2 — Random shift

**Motivation:** the aligned grid always places block boundaries at the same positions; a random per-iteration shift might diversify the holdout structure.

**Implementation:** each iteration draws a random offset in [0, L) and shifts the entire grid: `offset, offset+L, offset+2L, …`.

**Finding:** this made uniformity *worse* (std 0.0441 vs 0.0158). The reason is a boundary effect: when offset > 0, the shifted grid doesn't reach position 0 or the last few positions, so those edge positions are never held out in most iterations. The aligned v1 is already perfectly uniform — the shift only introduced asymmetry.

---

### v3 — Circular shift

**Motivation:** fix the v2 boundary effect by ensuring every position is always in exactly one block regardless of offset.

**Implementation:** instead of `np.arange(offset, n - L + 1, L)`, always generate exactly `n_full_blocks` blocks using `circular_starts = (np.arange(n_full_blocks) * L + offset) % n`. Positions are assigned to blocks modulo n, so one block wraps around from the high-energy end to the low-energy end when offset > 0.

**Trade-off:** the wrap-around block mixes high- and low-energy points, which is physically meaningless for a spectrum. This was accepted as the cost of uniform coverage.

**Finding:** best holdout uniformity of all five versions (std 0.0122).

---

### v4 — Random block lengths

**Motivation:** rather than shifting a fixed-length grid, vary the block lengths themselves so holdout boundaries fall at different positions each iteration.

**Implementation:** each iteration partitions [0, n) into blocks of lengths drawn uniformly from [6, 10], then holds out ~1/3 of those blocks. Bootstrap resampling uses fixed length 6 (the minimum) to keep interface compatibility.

**Finding:** moderate uniformity (std 0.0184), and noticeably wider PE CI than v1 and v3. The wider CI reflects extra iteration-to-iteration variance: when random block lengths don't sum cleanly to n, the holdout fraction fluctuates slightly, introducing noise in the PE estimates beyond what the other versions produce.

---

### v5 — Alternating direction

**Motivation:** in v4, the truncated block always falls at the high-energy (right) end of the spectrum. Reversing every other holdout mask should distribute the truncation bias symmetrically between both ends.

**Implementation:** after building the v4 holdout mask, flip it for odd iterations: `holdout_mask = holdout_mask[::-1]`.

**Finding:** slightly better uniformity than v4 (std 0.0148 vs 0.0184), confirming the reversal is doing its job. However, both remain dominated by v3, and the PE CI is the widest of all five versions.

---

### Final results

| Version | Holdout freq std | PE mean | Mean 95% CI | PE median | Median 95% CI |
|---|---|---|---|---|---|
| v1 (aligned) | 0.0158 | 0.028588 | [0.028292, 0.028902] | 0.028044 | [0.027745, 0.028416] |
| v2 (shifted) | 0.0441 | 0.029023 | [0.028687, 0.029366] | 0.028288 | [0.028056, 0.028584] |
| **v3 (circular)** | **0.0122** | **0.028914** | **[0.028527, 0.029341]** | **0.028128** | **[0.027870, 0.028408]** |
| v4 (random lengths) | 0.0184 | 0.029202 | [0.028806, 0.029624] | 0.028376 | [0.028117, 0.028675] |
| v5 (alternating) | 0.0148 | 0.029423 | [0.029019, 0.029852] | 0.028339 | [0.027968, 0.028662] |

---

### Analysis

**The CIs overlap substantially across all five versions.** CI widths are narrow (0.0003–0.0008), and every version's CI for mean and median overlaps with every other version's. This means no version produces a meaningfully different PE estimate — the choice between versions is driven entirely by holdout uniformity, not PE magnitude.

**CI width tracks with PE std.** The mean CIs are noticeably wider than the median CIs for all versions, consistent with the mean being more sensitive to the heavy right tail of the PE distributions. v4 and v5 have the widest CIs overall, reflecting the extra iteration-to-iteration variance introduced by random block lengths.

**v3 is the strongest design overall.** It achieves the best holdout uniformity by a clear margin (0.0122 vs v1's 0.0158) and its PE CI is competitive with v1. The only cost is one wrap-around block per iteration that mixes high- and low-energy points — a physically artificial construct that apparently has no measurable effect on PE estimates.

**v1 remains a strong choice.** Its PE CI is the narrowest of the five, its structure is the simplest, and its uniformity (while second-best) is genuine: every position is covered exactly once per iteration with equal probability. If the wrap-around block in v3 is considered unacceptable for physical reasons, v1 is the natural fallback.

**v2, v4, and v5 are not recommended.** v2 is strictly dominated by v1 on both metrics. v4 and v5 add implementation complexity without improving either uniformity or PE CI width relative to v1 or v3.


In [ ]:

# Compare v1–v5

import matplotlib.gridspec as gridspec

coef_cmp, fitted_cmp, residuals_cmp = fit_nnls(A, b)
n_cmp = len(b)
n_bootstrap_cmp = 1000

holdout_masks_v1, sampled_starts_v1, block_length_v1, _ = \
    select_holdout_blocks(n_cmp, np.random.default_rng(seed=0), n_bootstrap=n_bootstrap_cmp)
holdout_masks_v2, sampled_starts_v2, block_length_v2, _ = \
    select_holdout_blocks_v2(n_cmp, np.random.default_rng(seed=0), n_bootstrap=n_bootstrap_cmp)
holdout_masks_v3, sampled_starts_v3, block_length_v3, _ = \
    select_holdout_blocks_v3(n_cmp, np.random.default_rng(seed=0), n_bootstrap=n_bootstrap_cmp)
holdout_masks_v4, sampled_starts_v4, block_length_v4, _ = \
    select_holdout_blocks_v4(n_cmp, np.random.default_rng(seed=0), n_bootstrap=n_bootstrap_cmp)
holdout_masks_v5, sampled_starts_v5, block_length_v5, _ = \
    select_holdout_blocks_v5(n_cmp, np.random.default_rng(seed=0), n_bootstrap=n_bootstrap_cmp)

_, bootstrap_pes_v1 = do_moving_block_holdout_bootstrap(
    A, b, fitted_cmp, residuals_cmp, holdout_masks_v1, sampled_starts_v1, block_length_v1,
)
_, bootstrap_pes_v2 = do_moving_block_holdout_bootstrap(
    A, b, fitted_cmp, residuals_cmp, holdout_masks_v2, sampled_starts_v2, block_length_v2,
)
_, bootstrap_pes_v3 = do_moving_block_holdout_bootstrap(
    A, b, fitted_cmp, residuals_cmp, holdout_masks_v3, sampled_starts_v3, block_length_v3,
)
_, bootstrap_pes_v4 = do_moving_block_holdout_bootstrap(
    A, b, fitted_cmp, residuals_cmp, holdout_masks_v4, sampled_starts_v4, block_length_v4,
)
_, bootstrap_pes_v5 = do_moving_block_holdout_bootstrap(
    A, b, fitted_cmp, residuals_cmp, holdout_masks_v5, sampled_starts_v5, block_length_v5,
)

versions = [
    ('v1 (aligned)',        holdout_masks_v1.mean(axis=0), bootstrap_pes_v1, 'steelblue'),
    ('v2 (shifted)',        holdout_masks_v2.mean(axis=0), bootstrap_pes_v2, 'darkorange'),
    ('v3 (circular shift)', holdout_masks_v3.mean(axis=0), bootstrap_pes_v3, 'crimson'),
    ('v4 (random lengths)', holdout_masks_v4.mean(axis=0), bootstrap_pes_v4, 'mediumseagreen'),
    ('v5 (alternating)',    holdout_masks_v5.mean(axis=0), bootstrap_pes_v5, 'mediumpurple'),
]

# Pre-compute 95% CIs (done once so plots and printed table are consistent)
def bootstrap_ci(values, stat_fn, n_resamples=5000, rng=None):
    if rng is None:
        rng = np.random.default_rng()
    idx = rng.integers(0, len(values), size=(n_resamples, len(values)))
    stats = stat_fn(values[idx], axis=1)
    return np.percentile(stats, [2.5, 97.5])

ci_rng = np.random.default_rng(seed=1)
mean_cis   = [bootstrap_ci(pes, np.mean,   rng=ci_rng) for _, _, pes, _ in versions]
median_cis = [bootstrap_ci(pes, np.median, rng=ci_rng) for _, _, pes, _ in versions]

# Layout: row 0 = frequency plots, row 1 = hist + violin, row 2 = CI plots
fig = plt.figure(figsize=(25, 15))
gs = gridspec.GridSpec(3, 5, figure=fig, hspace=0.45)
ax_freqs     = [fig.add_subplot(gs[0, j]) for j in range(5)]
ax_hist      = fig.add_subplot(gs[1, :4])
ax_violin    = fig.add_subplot(gs[1, 4])
ax_mean_ci   = fig.add_subplot(gs[2, :2])
ax_median_ci = fig.add_subplot(gs[2, 3:])

# Row 0: holdout frequency per energy position
all_freq_vals = np.concatenate([v[1] for v in versions])
freq_ymin = all_freq_vals.min() * 0.9
freq_ymax = all_freq_vals.max() * 1.1

for ax, (label, freq, pes, color) in zip(ax_freqs, versions):
    ax.plot(energies, freq, color=color, linewidth=0.8)
    ax.axhline(freq.mean(), color='red', linestyle='--', label=f'mean={freq.mean():.3f}')
    ax.set_xlabel('Energy (eV)')
    ax.set_ylabel('Fraction held out')
    ax.set_title(label, fontsize=10)
    ax.set_ylim(freq_ymin, freq_ymax)
    ax.legend(fontsize=8)

# Row 1 left: overlaid PE histograms
for label, freq, pes, color in versions:
    ax_hist.hist(pes, bins=40, alpha=0.5, color=color, label=f'{label} mean={pes.mean():.5f}', density=True)
    ax_hist.axvline(pes.mean(), color=color, linestyle='--', linewidth=1.5)
ax_hist.set_xlabel('Prediction Error (RMSE)')
ax_hist.set_ylabel('Density')
ax_hist.set_title('Holdout prediction error distributions')
ax_hist.legend(fontsize=9)

# Row 1 right: violin comparison
parts = ax_violin.violinplot([v[2] for v in versions], positions=range(1, 6), showmedians=True)
for pc, (_, _, _, color) in zip(parts['bodies'], versions):
    pc.set_facecolor(color)
    pc.set_alpha(0.7)
ax_violin.set_xticks(range(1, 6))
ax_violin.set_xticklabels([f'v{i}' for i in range(1, 6)], fontsize=9)
ax_violin.set_ylabel('Prediction Error (RMSE)')
ax_violin.set_title('PE violin comparison')

# Shared y-axis limits for the two CI plots
all_ci_bounds = [b for lo, hi in mean_cis + median_cis for b in (lo, hi)]
ci_ymin = min(all_ci_bounds) * 0.9995
ci_ymax = max(all_ci_bounds) * 1.0005

# Row 2 left: 95% CI of mean PE
for j, ((label, _, pes, color), (lo, hi)) in enumerate(zip(versions, mean_cis)):
    ax_mean_ci.errorbar(
        j + 1, pes.mean(),
        yerr=[[pes.mean() - lo], [hi - pes.mean()]],
        fmt='o', color=color, capsize=5, linewidth=1.5, markersize=7, label=label,
    )
ax_mean_ci.set_xticks(range(1, 6))
ax_mean_ci.set_xticklabels([f'v{i}' for i in range(1, 6)])
ax_mean_ci.set_ylabel('Prediction Error (RMSE)')
ax_mean_ci.set_title('95% CI of Mean Prediction Error')
ax_mean_ci.set_ylim(ci_ymin, ci_ymax)
ax_mean_ci.legend(fontsize=8)

# Row 2 right: 95% CI of median PE
for j, ((label, _, pes, color), (lo, hi)) in enumerate(zip(versions, median_cis)):
    ax_median_ci.errorbar(
        j + 1, np.median(pes),
        yerr=[[np.median(pes) - lo], [hi - np.median(pes)]],
        fmt='s', color=color, capsize=5, linewidth=1.5, markersize=7, label=label,
    )
ax_median_ci.set_xticks(range(1, 6))
ax_median_ci.set_xticklabels([f'v{i}' for i in range(1, 6)])
ax_median_ci.set_ylabel('Prediction Error (RMSE)')
ax_median_ci.set_title('95% CI of Median Prediction Error')
ax_median_ci.set_ylim(ci_ymin, ci_ymax)
ax_median_ci.legend(fontsize=8)

plt.suptitle(
    f'select_holdout_blocks v1–v5 — {n_bootstrap_cmp} iterations',
    fontsize=13,
)
plt.tight_layout(rect=[0, 0, 1, 0.97])
plt.show()

print('Prediction error summary (95% CI computed by resampling the bootstrap PE draws):')
for (label, freq, pes, color), (mean_lo, mean_hi), (median_lo, median_hi) \
        in zip(versions, mean_cis, median_cis):
    print(f'  {label}:')
    print(f'    mean={pes.mean():.6f}  95% CI [{mean_lo:.6f}, {mean_hi:.6f}]')
    print(f'    median={np.median(pes):.6f}  95% CI [{median_lo:.6f}, {median_hi:.6f}]')

print('\nHoldout frequency std across positions (lower = more uniform):')
for label, freq, pes, color in versions:
    print(f'  {label}: {freq.std():.4f}')


In [ ]:
from itertools import combinations
from math import comb


def do_ref_subsets_moving_block_holdout_bootstrap(b, A, M, rng, select_holdout_blocks_fn,
                                                  n_bootstrap=1000):
    """Bootstrap prediction error for every combination of references at each size in M.

    select_holdout_blocks_fn is called once and its draws are shared across all
    combinations so that per-combination prediction errors are directly comparable.

    Parameters
    ----------
    b                        : (n,) observed spectrum normalized fluorescence values
    A                        : (n, n_refs) interpolated reference spectrum fluorescence values
    M                        : list of combination sizes to evaluate (each between 1 and n_refs)
    rng                      : numpy random Generator
    select_holdout_blocks_fn : callable with signature (n, rng, n_bootstrap=...) ->
                               (holdout_masks, sampled_starts, block_length, n_holdout_blocks)
    n_bootstrap              : number of bootstrap iterations

    Returns
    -------
    results : dict of numpy arrays, one row per combination across all sizes in M:
        'M'               : (n_combinations,) int
        'ref_indices'     : (n_combinations, max_M) float — column indices into A, NaN for unused
        'bootstrap_coefs' : (n_combinations, n_bootstrap, max_M) float — NaN for unused coefficients
        'bootstrap_pes'   : (n_combinations, n_bootstrap) float
        'coef'            : (n_combinations, max_M) float — full-data NNLS coefficients, NaN for unused
        'fitted'          : (n_combinations, n) float
        'residuals'       : (n_combinations, n) float
        'lags'            : (n_lags+1,) int — ACF lag indices (shared across all combinations)
        'acf_values'      : (n_combinations, n_lags+1) float — ACF of full-data residuals
    holdout_masks    : (n_bootstrap, n) bool array
    sampled_starts   : (n_bootstrap, n_blocks_needed) int array
    block_length     : int
    n_holdout_blocks : int
    """
    n, n_refs = A.shape
    for m in M:
        if not (1 <= m <= n_refs):
            raise ValueError(f'M value {m} must be between 1 and n_refs={n_refs}')

    max_M = max(M)
    combos = [(m, ref_indices) for m in sorted(M) for ref_indices in combinations(range(n_refs), m)]
    n_combinations = len(combos)
    n_lags = min(40, n // 2)

    all_M = np.zeros(n_combinations, dtype=int)
    all_ref_indices = np.full((n_combinations, max_M), np.nan)
    all_coef = np.full((n_combinations, max_M), np.nan)
    all_fitted = np.zeros((n_combinations, n))
    all_residuals = np.zeros((n_combinations, n))

    for i, (m, ref_indices) in enumerate(combos):
        coef, fitted, residuals = fit_nnls(A[:, ref_indices], b)
        all_M[i] = m
        all_ref_indices[i, :m] = ref_indices
        all_coef[i, :m] = coef
        all_fitted[i] = fitted
        all_residuals[i] = residuals

    all_acf_values = np.zeros((n_combinations, n_lags + 1))
    ci_95 = 1.96 / np.sqrt(n)
    max_sig_lags = []

    for i in range(n_combinations):
        lags, acf_values = calculate_acf(all_residuals[i])
        all_acf_values[i] = acf_values
        sig_mask = np.abs(acf_values[1:]) > ci_95
        max_sig_lags.append(int(np.where(sig_mask)[0][-1]) + 1 if sig_mask.any() else 0)

    print(f'Significant autocorrelation lags across all subsets: {min(max_sig_lags)}–{max(max_sig_lags)}')

    holdout_masks, sampled_starts, block_length, n_holdout_blocks = \
        select_holdout_blocks_fn(n, rng, n_bootstrap=n_bootstrap)

    all_bootstrap_coefs = np.full((n_combinations, n_bootstrap, max_M), np.nan)
    all_bootstrap_pes = np.zeros((n_combinations, n_bootstrap))

    for i, (m, ref_indices) in enumerate(combos):
        bootstrap_coefs, bootstrap_pes = do_moving_block_holdout_bootstrap(
            A[:, ref_indices], b, all_fitted[i], all_residuals[i],
            holdout_masks, sampled_starts, block_length,
        )
        all_bootstrap_coefs[i, :, :m] = bootstrap_coefs
        all_bootstrap_pes[i] = bootstrap_pes

    results = {
        'M': all_M,
        'ref_indices': all_ref_indices,
        'bootstrap_coefs': all_bootstrap_coefs,
        'bootstrap_pes': all_bootstrap_pes,
        'coef': all_coef,
        'fitted': all_fitted,
        'residuals': all_residuals,
        'lags': lags,
        'acf_values': all_acf_values,
    }
    return results, holdout_masks, sampled_starts, block_length, n_holdout_blocks


In [ ]:
def plot_worst_median_best_pe_distributions(results, spectrum_name, ax):
    from matplotlib.patches import Patch

    unique_M = sorted(set(results['M']))
    n_M = len(unique_M)
    colors = plt.cm.tab10(np.linspace(0, 0.4, n_M))
    M_to_color = {m: c for m, c in zip(unique_M, colors)}

    violin_pes = []
    violin_positions = []
    violin_color_list = []
    violin_labels = []

    pos = 0
    for m in unique_M:
        m_indices = np.where(results['M'] == m)[0]
        medians_m = np.array([np.median(results['bootstrap_pes'][i]) for i in m_indices])
        asc = np.argsort(medians_m)  # ascending: asc[0] = best (lowest median PE)
        n_m = len(asc)

        if n_m == 1:
            picks = [(asc[0], 'only')]
        elif n_m == 2:
            picks = [(asc[-1], 'worst'), (asc[0], 'best')]
        else:
            picks = [(asc[-1], 'worst'), (asc[n_m // 2], 'median'), (asc[0], 'best')]

        for local_idx, rank_label in picks:
            violin_pes.append(results['bootstrap_pes'][m_indices[local_idx]])
            violin_positions.append(pos)
            violin_color_list.append(M_to_color[m])
            violin_labels.append(rank_label)
            pos += 1

        pos += 1  # gap between M groups

    parts = ax.violinplot(violin_pes, positions=violin_positions, showmedians=True)
    for idx, color in enumerate(violin_color_list):
        parts['bodies'][idx].set_facecolor(color)
        parts['bodies'][idx].set_alpha(0.7)

    ax.set_xticks(violin_positions)
    ax.set_xticklabels(violin_labels, fontsize=8)
    ax.set_ylabel('Holdout Prediction Error (RMSE)')
    ax.set_xlabel('Reference Subset')
    ax.set_title(f'Bootstrap Prediction Error — Worst / Median / Best per Subset Size — {spectrum_name}')

    legend_elements = [Patch(facecolor=M_to_color[m], alpha=0.7, label=f'M={m}') for m in unique_M]
    ax.legend(handles=legend_elements)


def plot_descending_median_pe(results, axs):
    unique_M = sorted(set(results['M']))
    n_M = len(unique_M)
    colors = plt.cm.tab10(np.linspace(0, 0.4, n_M))
    M_to_color = {m: c for m, c in zip(unique_M, colors)}

    all_pe_vals = []
    for j, m in enumerate(unique_M):
        m_indices = np.where(results['M'] == m)[0]
        m_order = m_indices[np.argsort(np.median(results['bootstrap_pes'][m_indices], axis=1))[::-1]]
        medians = np.median(results['bootstrap_pes'][m_order], axis=1)
        all_pe_vals.extend(medians.tolist())

        axs[j].plot(range(len(m_order)), medians, 's-', color=M_to_color[m], linewidth=1.5, label='median')
        axs[j].set_xlabel('Reference Subset')
        axs[j].set_title(f'M={m}')
        axs[j].legend(fontsize=8)

    pe_ymin = min(all_pe_vals) * 0.999
    pe_ymax = max(all_pe_vals) * 1.001
    for ax in axs:
        ax.set_ylim(pe_ymin, pe_ymax)
    axs[0].set_ylabel('Median Prediction Error (RMSE)')


def plot_ref_subsets_summary(
    results, ref_names, spectrum_name,
    sample_energies, interp_energies, elapsed_time,
):
    import pandas as pd

    n_combinations = len(results['M'])
    unique_M = sorted(set(results['M']))
    n_M = len(unique_M)

    fig, ax = plt.subplots(figsize=(max(10, 5 * n_M), 5))
    plot_worst_median_best_pe_distributions(results, spectrum_name, ax)
    plt.tight_layout()
    plt.show()

    fig, axs = plt.subplots(1, n_M, figsize=(max(10, 5 * n_M), 4))
    plot_descending_median_pe(results, np.atleast_1d(axs))
    plt.tight_layout()
    plt.show()

    rows = {
        'Sample':                      spectrum_name,
        'Sample energy points':        len(sample_energies),
        'Sample energy range':         f'{sample_energies[0]:.1f}–{sample_energies[-1]:.1f} eV',
        'References':                  len(ref_names),
        'Interpolation energy range':  f'{interp_energies[0]:.1f}–{interp_energies[-1]:.1f} eV',
        'Interpolation energy points': len(interp_energies),
        **{f'Subsets (M={m})': int((results['M'] == m).sum()) for m in unique_M},
        'Total bootstrap time':        f'{elapsed_time:.1f} s',
        'Avg time per subset':         f'{elapsed_time / n_combinations:.2f} s',
    }
    display(pd.DataFrame({'Value': rows}))


In [ ]:
def _ordinal(n):
    if 11 <= n % 100 <= 13:
        suffix = 'th'
    else:
        suffix = {1: 'st', 2: 'nd', 3: 'rd'}.get(n % 10, 'th')
    return f'{n}{suffix}'


def plot_best_subset_bootstrap_summaries(results, ref_names, energies, b, n_bootstrap, spectrum_name, N=3):
    """Call plot_bootstrap_summary for the top N best subsets of each subset size.

    "Best" is defined as lowest median bootstrap prediction error.

    Parameters
    ----------
    results       : dict returned by do_ref_subset_moving_block_holdout_bootstrap
    ref_names     : list of reference spectrum names
    energies      : energy grid array
    b             : observed spectrum values
    n_bootstrap   : number of bootstrap iterations (passed to plot_bootstrap_summary)
    spectrum_name : str — used in plot titles
    N             : number of best subsets per subset size to plot (default 3)
    """
    for m in sorted(set(results['M'])):
        m_indices = np.where(results['M'] == m)[0]
        medians_m = np.array([np.median(results['bootstrap_pes'][i]) for i in m_indices])
        for rank, local_idx in enumerate(np.argsort(medians_m)[:N], start=1):
            i = m_indices[local_idx]
            ref_idx = [int(j) for j in results['ref_indices'][i, :m]]
            subset_ref_names = [ref_names[j] for j in ref_idx]
            title_prefix = f'{_ordinal(rank)} best {m}-component fit'
            plot_bootstrap_summary(
                energies, b,
                results['fitted'][i],
                results['residuals'][i],
                results['lags'],
                results['acf_values'][i],
                results['bootstrap_coefs'][i, :, :m],
                results['bootstrap_pes'][i],
                results['coef'][i, :m],
                subset_ref_names,
                spectrum_name=spectrum_name,
                n_bootstrap=n_bootstrap,
                title_prefix=title_prefix,
            )

In [ ]:
import time

def plot_compare_select_holdout_blocks_functions(sample_spectrum, reference_spectra):
    n_bootstrap = 1000
    
    valid_energies, interpolated_ref_spectra, b, *_ = interpolate_references_at_sample_energies(
        reference_spectra=reference_spectra,
        sample_spectrum=sample_spectrum
    )
    
    holdout_fn_versions = [
        ('v1 (aligned)',        select_holdout_blocks),
        ('v2 (shifted)',        select_holdout_blocks_v2),
        ('v3 (circular shift)', select_holdout_blocks_v3),
        ('v4 (random lengths)', select_holdout_blocks_v4),
        ('v5 (alternating)',    select_holdout_blocks_v5),
    ]
    
    all_version_results = []
    for label, fn in holdout_fn_versions:
        print(f'\n=== {label} ===')
        t0 = time.perf_counter()
        results, holdout_masks, sampled_starts, block_length, n_holdout_blocks = \
            do_ref_subsets_moving_block_holdout_bootstrap(
                b, interpolated_ref_spectra, M=[1, 2, 3], rng=np.random.default_rng(seed=42),
                select_holdout_blocks_fn=fn,
                n_bootstrap=n_bootstrap,
            )
        elapsed = time.perf_counter() - t0
        all_version_results.append((label, results, elapsed))
    
    unique_M = sorted(set(all_version_results[0][1]['M']))
    n_M = len(unique_M)
    
    for label, results, elapsed in all_version_results:
        fig, ax = plt.subplots(figsize=(max(10, 5 * n_M), 5))
        plot_worst_median_best_pe_distributions(results, f'{sample_spectrum.file_name} [{label}]', ax)
        plt.tight_layout()
        plt.show()

sample_spectrum = filter_spectra_by_name(sample_spectra_list, "OTT3_55*")[0]
print(f'sample spectrum: {sample_spectrum.file_name}')
reference_spectra = filter_spectra_by_name(
    reference_spectra_list,
    "Arsenopyrite_Jul*", "orpiment_all*", "arsenate*_diop*", "Fh2l**"
)
ref_names = [r.file_name for r in reference_spectra]
print(f"reference spectra:\n{chr(10).join(chr(9) + r for r in ref_names)}")
    
plot_compare_select_holdout_blocks_functions(sample_spectrum, reference_spectra)

In [ ]:
import time

def do_fits_and_plot_summaries(sample_spectrum, reference_spectra):
    print(f'sample spectrum: {sample_spectrum.file_name}')
    ref_names = [r.file_name for r in reference_spectra]
    print(f"reference spectra:\n{chr(10).join(chr(9) + r for r in ref_names)}")
    
    n_bootstrap = 1000

    valid_energies_24, interpolated_ref_spectra, b, *_ = interpolate_references_at_sample_energies(
        reference_spectra=reference_spectra,
        sample_spectrum=sample_spectrum
    )
    
    t0 = time.perf_counter()
    results, holdout_masks, sampled_starts, block_length, n_holdout_blocks = \
        do_ref_subsets_moving_block_holdout_bootstrap(
            b, interpolated_ref_spectra, M=[1, 2, 3], rng=np.random.default_rng(seed=42),
            select_holdout_blocks_fn=select_holdout_blocks_v3,
            n_bootstrap=n_bootstrap,
        )
    elapsed = time.perf_counter() - t0
    
    plot_ref_subsets_summary(
        results, ref_names, spectrum_name=sample_spectrum.file_name,
        sample_energies=sample_spectrum.data_df.index.values,
        interp_energies=valid_energies_24,
        elapsed_time=elapsed,
    )
    
    plot_best_subset_bootstrap_summaries(
        results, ref_names, valid_energies_24, b, n_bootstrap,
        spectrum_name=sample_spectrum.file_name,
    )

sample_spectrum = filter_spectra_by_name(sample_spectra_list, "OTT3_55*")[0]
reference_spectra = filter_spectra_by_name(
    reference_spectra_list,
    "Arsenopyrite_Jul*", "orpiment_all*", "arsenate*_diop*", "*"
)

do_fits_and_plot_summaries(sample_spectrum, reference_spectra)

## Comparing interpolation methods: linear vs. cubic spline

Every reference spectrum is measured on its own energy grid, so all of them have to be
resampled onto the sample's grid before any fitting can happen. That resampling is the
very first step in the pipeline, which makes the interpolation method an assumption
sitting underneath the design matrix, the NNLS coefficients, the prediction error, and
ultimately the reference combination reported as the best fit.

Until now the method was fixed: `ReferenceSpectrum` builds an
`InterpolatedUnivariateSpline` (a cubic spline) at construction time, and
`interpolate_references_at_sample_energies` simply used it. `make_interpolant` now makes
that choice explicit and selectable, and this section measures what the choice is worth.

The comparison is deliberately controlled. The common energy range depends only on the
measured ranges of the sample and references, never on how they are interpolated, so
both arms fit the same `n` points and the same response vector `b`. Because `n` matches,
each arm's freshly seeded generator draws *identical* holdout masks — every bootstrap
iteration trains and scores on exactly the same positions in both arms. The design
matrix is the only thing that differs, and `compare_interpolation_methods` asserts all
three invariants rather than assuming them.

In [ ]:
def compare_interpolation_methods(
    sample_spectrum, reference_spectra, M=(1, 2, 3), n_bootstrap=1000, seed=42,
    select_holdout_blocks_fn=select_holdout_blocks_v3,
):
    """Run the full subset bootstrap twice, once per interpolation method.

    The two arms are matched as tightly as possible so that any difference in the
    results is attributable to the interpolation method alone:

    * The common energy range depends only on the measured energy ranges of the
      sample and references, never on how they are interpolated, so both arms get
      the same valid_energies, the same n, and the same response vector b.
    * Because n is identical, select_holdout_blocks_fn draws *identical* holdout
      masks and resample block starts from a freshly seeded generator in each arm.
      Every bootstrap iteration therefore trains and scores on exactly the same
      positions in both arms, and the only thing that differs is the design matrix.

    Both invariants are asserted below rather than assumed.

    Returns
    -------
    comparison : dict with keys 'valid_energies', 'b', 'ref_names', and one entry per
                 method name ('linear', 'cubic') holding {'A', 'results', 'elapsed'}
    """
    import time

    methods = [
        ('linear', make_linear_interpolant),
        ('cubic', make_cubic_spline_interpolant),
    ]

    comparison = {
        'ref_names': [r.file_name for r in reference_spectra],
        # the references' own measured points, kept so the comparison plot can show
        # the nodes the two interpolants are drawn through
        'reference_energies': [r.data_df.index.values for r in reference_spectra],
        'reference_norm': [r.data_df['norm'].values for r in reference_spectra],
    }
    holdout_masks_by_method = {}

    for method_name, make_interpolant in methods:
        print(f'\n=== {method_name} ===')
        valid_energies, A, b, *_ = interpolate_references_at_sample_energies(
            reference_spectra=reference_spectra,
            sample_spectrum=sample_spectrum,
            make_interpolant=make_interpolant,
        )

        t0 = time.perf_counter()
        results, holdout_masks, _, _, _ = do_ref_subsets_moving_block_holdout_bootstrap(
            b, A, M=list(M), rng=np.random.default_rng(seed=seed),
            select_holdout_blocks_fn=select_holdout_blocks_fn,
            n_bootstrap=n_bootstrap,
        )
        elapsed = time.perf_counter() - t0

        if 'valid_energies' in comparison:
            # the controlled-comparison invariants
            np.testing.assert_array_equal(comparison['valid_energies'], valid_energies)
            np.testing.assert_array_equal(comparison['b'], b)
        comparison['valid_energies'] = valid_energies
        comparison['b'] = b
        comparison[method_name] = {'A': A, 'results': results, 'elapsed': elapsed}
        holdout_masks_by_method[method_name] = holdout_masks

    np.testing.assert_array_equal(
        holdout_masks_by_method['linear'], holdout_masks_by_method['cubic'],
    )
    print('\nboth arms used identical energies, response vector, and holdout masks')

    return comparison


def summarize_interpolation_comparison(comparison, top_n=10):
    """Print the headline comparison: does the interpolation method change the answer?"""
    import pandas as pd
    from scipy.stats import spearmanr

    A_linear = comparison['linear']['A']
    A_cubic = comparison['cubic']['A']
    ref_names = comparison['ref_names']

    print('Design matrix')
    print(f'  max |cubic - linear| : {np.abs(A_cubic - A_linear).max():.5f} norm units')
    print(f'  rms |cubic - linear| : {np.sqrt(np.mean((A_cubic - A_linear) ** 2)):.5f} norm units')
    worst = np.argsort(np.abs(A_cubic - A_linear).max(axis=0))[::-1][:5]
    print('  largest-disagreeing references:')
    for i in worst:
        print(f'    {ref_names[i]}: max |diff| {np.abs(A_cubic[:, i] - A_linear[:, i]).max():.5f}')

    median_pe = {
        method: np.median(comparison[method]['results']['bootstrap_pes'], axis=1)
        for method in ('linear', 'cubic')
    }
    results = comparison['cubic']['results']

    print('\nPrediction error over all reference combinations')
    rho, _ = spearmanr(median_pe['linear'], median_pe['cubic'])
    print(f'  combinations: {len(results["M"])}')
    print(f'  Spearman rank correlation of median PE: {rho:.5f}')
    for method in ('linear', 'cubic'):
        print(f'  {method:6s}: median PE over combinations = {np.median(median_pe[method]):.6f}, '
              f'best = {median_pe[method].min():.6f}, {comparison[method]["elapsed"]:.1f}s')

    def subset_names(index):
        ref_indices = results['ref_indices'][index]
        return ' + '.join(ref_names[int(j)] for j in ref_indices[~np.isnan(ref_indices)])

    print('\nBest subset by median PE, per subset size')
    rows = []
    for m in sorted(set(results['M'])):
        m_indices = np.where(results['M'] == m)[0]
        best = {
            method: m_indices[np.argmin(median_pe[method][m_indices])]
            for method in ('linear', 'cubic')
        }
        rows.append({
            'M': m,
            'best (linear)': subset_names(best['linear']),
            'best (cubic)': subset_names(best['cubic']),
            'same?': 'yes' if best['linear'] == best['cubic'] else 'NO',
            'PE linear': f'{median_pe["linear"][best["linear"]]:.6f}',
            'PE cubic': f'{median_pe["cubic"][best["cubic"]]:.6f}',
        })
    display(pd.DataFrame(rows).set_index('M'))

    print(f'\nTop-{top_n} agreement (by median PE, across all subset sizes)')
    top = {method: set(np.argsort(median_pe[method])[:top_n]) for method in ('linear', 'cubic')}
    shared = top['linear'] & top['cubic']
    print(f'  {len(shared)} of {top_n} subsets appear in both top-{top_n} lists')
    overall_best = {method: int(np.argmin(median_pe[method])) for method in ('linear', 'cubic')}
    print(f'  overall best (linear): {subset_names(overall_best["linear"])}')
    print(f'  overall best (cubic) : {subset_names(overall_best["cubic"])}')
    print(f'  overall best agrees: '
          f'{"yes" if overall_best["linear"] == overall_best["cubic"] else "NO"}')

    return median_pe

In [ ]:
def plot_interpolation_method_comparison(comparison, median_pe, spectrum_name, n_emphasize=4):
    """Visualize how linear vs. cubic spline interpolation propagates to the results.

    Six panels, reading top-left to bottom-right as the causal chain: where the two
    design matrices differ, why they differ there, and whether that difference
    survives into the prediction errors and the selected reference subset.
    """
    import matplotlib.gridspec as gridspec
    from matplotlib.patches import Patch
    from matplotlib.ticker import NullFormatter
    from scipy.stats import spearmanr

    energies = comparison['valid_energies']
    A_linear = comparison['linear']['A']
    A_cubic = comparison['cubic']['A']
    ref_names = comparison['ref_names']
    results = comparison['cubic']['results']
    difference = A_cubic - A_linear

    unique_M = sorted(set(results['M']))
    colors = plt.cm.tab10(np.linspace(0, 0.4, len(unique_M)))
    M_to_color = {m: c for m, c in zip(unique_M, colors)}

    # rank references by how much the two methods disagree about them
    per_reference_max = np.abs(difference).max(axis=0)
    emphasized = np.argsort(per_reference_max)[::-1][:n_emphasize]
    # explicit hues rather than a colormap slice: the un-emphasized references are
    # gray, so an emphasis color that lands near gray would be invisible
    emphasis_colors = ['crimson', 'darkorange', 'seagreen', 'mediumpurple',
                       'steelblue', 'saddlebrown'][:n_emphasize]

    fig = plt.figure(figsize=(21, 14))
    gs = gridspec.GridSpec(3, 3, figure=fig, hspace=0.35, wspace=0.25)

    # ---------------------------------------------------------------- panel 1
    # Where the methods disagree, per reference, across the fitted energy range.
    ax = fig.add_subplot(gs[0, :2])
    for i in range(difference.shape[1]):
        if i not in emphasized:
            ax.plot(energies, difference[:, i], color='0.75', linewidth=0.7, zorder=1)
    for rank, i in enumerate(emphasized):
        ax.plot(
            energies, difference[:, i], color=emphasis_colors[rank], linewidth=1.3, zorder=3,
            label=f'{ref_names[i]} (max {per_reference_max[i]:.4f})',
        )
    ax.axhline(0, color='black', linewidth=0.8, zorder=2)
    ax.plot([], [], color='0.75', linewidth=0.7,
            label=f'other references ({difference.shape[1] - n_emphasize})')
    ax.set_xlabel('Energy (eV)')
    ax.set_ylabel('cubic − linear (norm units)')
    ax.set_title('Where the two interpolations disagree, per reference', fontsize=11)
    ax.legend(fontsize=7, loc='upper right')

    # ---------------------------------------------------------------- panel 2
    # Why they disagree: the worst reference's own measured points, zoomed on the
    # window where the disagreement peaks, with both interpolants drawn through them.
    ax = fig.add_subplot(gs[0, 2])
    worst_reference = int(emphasized[0])
    peak_energy = energies[np.abs(difference[:, worst_reference]).argmax()]
    # a tight window: over any wider span the two interpolants are visually identical
    # and the panel says nothing. The disagreement is a local, few-eV effect.
    window = 2.0
    dense = np.linspace(peak_energy - window, peak_energy + window, 600)
    dense = dense[(dense >= energies[0]) & (dense <= energies[-1])]

    reference_spectrum_energies = comparison['reference_energies'][worst_reference]
    reference_spectrum_norm = comparison['reference_norm'][worst_reference]
    in_window = (
        (reference_spectrum_energies >= dense[0] - 2) & (reference_spectrum_energies <= dense[-1] + 2)
    )
    ax.plot(dense, make_cubic_spline_interpolant(
        reference_spectrum_energies, reference_spectrum_norm)(dense),
        color='crimson', linewidth=1.5, label='cubic spline')
    ax.plot(dense, make_linear_interpolant(
        reference_spectrum_energies, reference_spectrum_norm)(dense),
        color='steelblue', linewidth=1.5, linestyle='--', label='linear')
    ax.plot(
        reference_spectrum_energies[in_window], reference_spectrum_norm[in_window],
        'o', color='black', markersize=5, zorder=5, label='measured points',
    )
    ax.set_xlim(dense[0], dense[-1])
    # absolute eV, not an offset like '+1.187e4' -- the window is only a few eV wide
    ax.ticklabel_format(useOffset=False, axis='x')
    ax.set_xlabel('Energy (eV)')
    ax.set_ylabel('norm')
    ax.set_title(
        f'{ref_names[worst_reference]}\nnode spacing '
        f'{np.median(np.diff(reference_spectrum_energies)):.2f} eV',
        fontsize=9,
    )
    ax.legend(fontsize=7)

    # ---------------------------------------------------------------- panel 3
    # Does the difference survive into prediction error? One point per reference
    # combination; the y=x line is where the method makes no difference at all.
    ax = fig.add_subplot(gs[1, 0])
    for m in unique_M:
        m_mask = results['M'] == m
        ax.scatter(
            median_pe['linear'][m_mask], median_pe['cubic'][m_mask],
            s=8, alpha=0.5, color=M_to_color[m], label=f'M={m}',
        )
    lims = [
        min(median_pe['linear'].min(), median_pe['cubic'].min()) * 0.98,
        max(median_pe['linear'].max(), median_pe['cubic'].max()) * 1.02,
    ]
    ax.plot(lims, lims, color='black', linewidth=1.0, linestyle=':', label='no difference')
    # log-log: prediction errors span an order of magnitude across combinations, and on
    # a linear scale the well-fitting combinations -- the only ones model selection can
    # ever choose between -- collapse into one corner
    ax.set_xscale('log')
    ax.set_yscale('log')
    # log minor ticks label every 2x/3x/4x decade step here and collide into mush
    ax.xaxis.set_minor_formatter(NullFormatter())
    ax.yaxis.set_minor_formatter(NullFormatter())
    ax.set_xlim(lims)
    ax.set_ylim(lims)
    ax.set_aspect('equal')
    rho, _ = spearmanr(median_pe['linear'], median_pe['cubic'])
    ax.set_xlabel('Median PE — linear')
    ax.set_ylabel('Median PE — cubic')
    ax.set_title(f'Median prediction error per combination\nSpearman ρ = {rho:.5f}', fontsize=10)
    ax.legend(fontsize=7)

    # ---------------------------------------------------------------- panel 4
    # Is one method systematically optimistic? A shift away from zero means the
    # choice biases the PE level, not just its ranking.
    ax = fig.add_subplot(gs[1, 1])
    delta = median_pe['cubic'] - median_pe['linear']
    for m in unique_M:
        ax.hist(delta[results['M'] == m], bins=40, alpha=0.6, color=M_to_color[m], label=f'M={m}')
    ax.axvline(0, color='black', linewidth=1.0, linestyle=':')
    ax.axvline(
        np.median(delta), color='red', linestyle='--', linewidth=1.2,
        label=f'median Δ = {np.median(delta):+.6f}',
    )
    ax.set_xlabel('Median PE: cubic − linear')
    ax.set_ylabel('Combinations')
    ax.set_title('Signed prediction error shift', fontsize=10)
    ax.legend(fontsize=7)

    # ---------------------------------------------------------------- panel 5
    # Ranking stability where it actually matters. Model selection only ever looks
    # at the best few combinations, so a high overall rank correlation can still
    # hide a reordering at the top.
    ax = fig.add_subplot(gs[1, 2])
    n_top = 30
    rank_linear = np.argsort(np.argsort(median_pe['linear']))
    rank_cubic = np.argsort(np.argsort(median_pe['cubic']))
    top_mask = (rank_cubic < n_top) | (rank_linear < n_top)
    for m in unique_M:
        m_mask = top_mask & (results['M'] == m)
        if not m_mask.any():
            continue  # subset sizes absent from the top N would add an empty legend row
        ax.scatter(
            rank_linear[m_mask], rank_cubic[m_mask],
            s=25, alpha=0.75, color=M_to_color[m], label=f'M={m}',
        )
    ax.plot([0, n_top], [0, n_top], color='black', linewidth=1.0, linestyle=':')
    ax.set_xlim(-1, n_top)
    ax.set_ylim(-1, n_top)
    ax.set_aspect('equal')
    ax.set_xlabel('Rank — linear (0 = best)')
    ax.set_ylabel('Rank — cubic (0 = best)')
    ax.set_title(f'Rank agreement among the top {n_top} combinations', fontsize=10)
    ax.legend(fontsize=7)

    # ---------------------------------------------------------------- panel 6
    # The practical output: the coefficients a user would report. Shown for the
    # subset each method selects as best overall, with bootstrap 95% intervals.
    ax = fig.add_subplot(gs[2, :])

    # Group by reference rather than by method, so the two methods' coefficients for
    # the same reference sit side by side and are directly comparable. Taking the union
    # of both selected subsets means this still reads correctly if the methods pick
    # different subsets -- a reference chosen by only one method simply has one bar.
    per_method = {}
    for method in ('linear', 'cubic'):
        best = int(np.argmin(median_pe[method]))
        method_results = comparison[method]['results']
        ref_indices = method_results['ref_indices'][best]
        ref_indices = ref_indices[~np.isnan(ref_indices)].astype(int)
        n_used = len(ref_indices)
        bootstrap_coefficients = method_results['bootstrap_coefs'][best][:, :n_used]
        per_method[method] = {
            'ref_indices': list(ref_indices),
            'coef': method_results['coef'][best][:n_used],
            'lo': np.percentile(bootstrap_coefficients, 2.5, axis=0),
            'hi': np.percentile(bootstrap_coefficients, 97.5, axis=0),
        }

    union_refs = list(dict.fromkeys(
        per_method['linear']['ref_indices'] + per_method['cubic']['ref_indices']
    ))
    same_subset = set(per_method['linear']['ref_indices']) == set(per_method['cubic']['ref_indices'])

    width = 0.36  # < half the 0.4 offset spacing, so the paired bars keep a visible gap
    for offset, (method, color) in zip((-0.20, 0.20),
                                       (('linear', 'steelblue'), ('cubic', 'crimson'))):
        entry = per_method[method]
        for position, ref_index in enumerate(union_refs):
            if ref_index not in entry['ref_indices']:
                continue  # this method did not select this reference
            k = entry['ref_indices'].index(ref_index)
            ax.bar(position + offset, entry['coef'][k], color=color, alpha=0.8, width=width)
            ax.errorbar(
                position + offset, entry['coef'][k],
                yerr=[[entry['coef'][k] - entry['lo'][k]], [entry['hi'][k] - entry['coef'][k]]],
                fmt='none', ecolor='black', capsize=4, linewidth=1.2,
            )

    ax.set_xticks(range(len(union_refs)))
    ax.set_xticklabels([ref_names[i] for i in union_refs], fontsize=8)
    ax.set_ylabel('NNLS coefficient')
    ax.set_title(
        'Best subset selected by each method — full-data coefficients with bootstrap 95% intervals'
        + ('  (both methods selected the same subset)' if same_subset
           else '  (METHODS SELECTED DIFFERENT SUBSETS)'),
        fontsize=11,
    )
    ax.legend(
        handles=[
            Patch(facecolor='steelblue', alpha=0.8, label='linear'),
            Patch(facecolor='crimson', alpha=0.8, label='cubic'),
        ],
        fontsize=8,
    )

    fig.suptitle(
        f'Linear vs. cubic spline interpolation of reference spectra — {spectrum_name}',
        fontsize=13, y=0.98,
    )
    fig.subplots_adjust(left=0.05, right=0.98, top=0.92, bottom=0.10)
    plt.show()

In [ ]:
sample_spectrum = filter_spectra_by_name(sample_spectra_list, "OTT3_55*")[0]
reference_spectra = filter_spectra_by_name(reference_spectra_list, "*")

comparison = compare_interpolation_methods(sample_spectrum, reference_spectra)
median_pe = summarize_interpolation_comparison(comparison)
plot_interpolation_method_comparison(comparison, median_pe, sample_spectrum.file_name)

### Findings

Run on `OTT3_55_spot0.e` against the full 24-reference pool, M = [1, 2, 3] (2,324
combinations), 1,000 bootstrap iterations, `select_holdout_blocks_v3`, seed 42. Each arm
took ~23.5 s.

#### The design matrices genuinely differ

| quantity | value |
|---|---|
| max &#124;cubic − linear&#124; | 0.05353 norm units |
| rms &#124;cubic − linear&#124; | 0.00382 norm units |

For scale, the best fit's prediction error is ~0.0280 RMSE, so the *peak* disagreement
between the two interpolations is nearly **twice** the error the model is judged by.
This is not a rounding-level difference.

The largest-disagreeing references:

| reference | max &#124;diff&#124; | node spacing |
|---|---|---|
| scorodite_A_Foster_sln.e | 0.05353 | 0.50 eV |
| arsenate_sorbed_anth_avg_als_cal.e | 0.05300 | 1.05 eV |
| arsenate_sorbed_diop_avg_als_cal.e | 0.04052 | 1.05 eV |
| arsenate_sorbed_opal_avg1_8_als_cal.e | 0.03604 | 1.05 eV |
| arsenate_sorbed_calcite_avg1_4_als_cal.e | 0.03312 | 1.05 eV |

**Curvature matters more than node spacing.** The expectation going in was that the six
references measured every 1.05 eV — coarser than the sample's 0.5 eV grid — would
dominate. They are indeed over-represented, but the single worst offender, `scorodite`,
sits on a perfectly ordinary 0.50 eV grid. Panel 1 shows why: the disagreement is
essentially zero above ~11900 eV and concentrates entirely in 11845–11880 eV, the
absorption edge and white line. Where a spectrum is nearly straight between nodes, a
chord and a spline agree no matter how far apart the nodes are; where it turns sharply,
they diverge. Panel 2 shows the mechanism directly — the linear interpolant cuts a chord
across the white-line peak while the cubic spline rounds over it.

#### But it does not change any conclusion

| quantity | linear | cubic |
|---|---|---|
| Spearman ρ of median PE (2,324 combinations) | 0.99992 | 0.99992 |
| median PE across all combinations | 0.139639 | 0.139952 |
| best median PE | 0.028235 | **0.028023** |
| best subset at M=1 | Lollingite_Lolling_avg_OA | *same* |
| best subset at M=2 | As2O3_ref_avg + As_pyrite_A_Foster | *same* |
| best subset at M=3 | Arsenopyrite_Julcani + arsenate_sorbed_diop + orpiment_all_ref | *same* |
| top-10 combinations | identical sets | identical sets |

Model selection is completely insensitive to the choice here. The best subset agrees at
every subset size, the ten best combinations overall are the same ten, and the rank
correlation across all 2,324 combinations is 0.99992. The fitted coefficients for the
selected M=3 subset are indistinguishable between the methods, well inside their
bootstrap 95% intervals (panel 6).

#### One asymmetry worth noting

Cubic is very slightly *worse* on the typical combination (median PE higher by
+0.000263) but slightly *better* at the optimum (0.028023 vs 0.028235). A plausible
reading is that where a combination genuinely explains the sample, the smoother cubic
references track the real spectrum more faithfully and shave a little error; where the
combination is a poor explanation, the interpolation difference is just one more
uncorrelated perturbation. The effect is small either way and this data cannot really
separate those explanations — it is offered as an observation, not a conclusion.

#### Recommendation

**Keep cubic spline as the default.** It matches what `ReferenceSpectrum` already builds
(and reproduces it to ~1e-15), it is marginally better at the selected optimum, and the
comparison shows nothing to be gained by switching. The useful result is the negative
one: the hardcoded spline in `mrfitty/base.py` is not quietly steering model selection
on this dataset.

That conclusion is dataset-specific in one identifiable way. The disagreement lives
entirely at the absorption edge, so a reference set measured coarsely *through the edge*
— rather than coarsely overall, as here — could behave differently. Re-running this
section is the way to check, which is why it lives in the notebook as a permanent,
re-runnable comparison rather than a one-off answer.